# TFM · Análisis multidimensional del estrés estudiantil
### Notebook único — Entrega 1 (datos y EDA) + Entrega 2 (ETL y KPIs)

> **Cómo ejecutarlo:** con XAMPP en marcha (MySQL en verde), menú **Kernel → Restart Kernel and Run All Cells**.
> No hay que ejecutar nada a mano. Se puede relanzar las veces que haga falta: no duplica datos.

El Alcance del TFM pide que todo el Python de la Entrega 2 esté **en el mismo notebook que el de la Entrega 1**. Este archivo reúne las dos entregas, en orden:

| Parte | Contenido | Qué hace |
|---|---|---|
| **A · Entrega 1** | EDA de los 4 bloques analíticos (12 datasets) | Perfilado, calidad, análisis univariante y bivariante, hallazgos y 13 figuras |
| **B.1 · Entrega 2** | Ingesta a MySQL | Carga los 12 CSV en las tablas del modelo v2 con sus claves conformes |
| **B.2 · Entrega 2** | ETL y KPIs | Transforma desde la estrella, carga el datawarehouse (`dwh_*`) y guarda 6 KPIs con histórico |

**Requisitos previos (una sola vez, en phpMyAdmin):** importar `modelo_datos_mysql_v2.sql` y después `esquema_estrella_powerbi.sql`. El datawarehouse (`dwh_estrella.sql`) no hace falta: la Parte B.2 lo crea sola si falta.

---
# PARTE A · Entrega 1 — Análisis exploratorio de datos (EDA)

Cada bloque sigue la misma plantilla de 5 pasos: **1** carga e inventario · **2** calidad (nulos, duplicados, anomalías) · **3** análisis univariante · **4** análisis bivariante · **5** hallazgos.
Las figuras se guardan en la carpeta `figuras1` junto a los datos. *El código de esta parte no se ha modificado.*

## Bloque 1 — Factores internos (núcleo del estrés)

4 datasets: Academic Stress, Anxiety GAD-7, Mental Health Survey y Student Habits. Tres de ellos tienen señal real; Student Habits resulta sintético (sin correlación entre hábitos y salud mental).

In [ ]:
# -*- coding: utf-8 -*-
"""
================================================================================
OBJETIVO 1 - ANALISIS EXPLORATORIO Y DESCRIPTIVO (EDA)
BLOQUE 1: FACTORES INTERNOS (CORE DEL ESTRES)
================================================================================
TFM: Analisis multidimensional del estres estudiantil mediante Business
     Analytics e Inteligencia Artificial

Datasets analizados (4):
  1.1 academic Stress level - maintainance 1.csv  (Kaggle: poushal02)
  1.2 anxiety.csv                                  (Kaggle: petalme)
  1.3 MentalHealthSurvey.csv                       (Kaggle: abdullahashfaqvirk)
  1.4 student_habits_performance.csv               (GitHub: abhi88377)

Metodologia aplicada a cada dataset (5 pasos):
  PASO 1 - Carga e inventario: dimensiones, tipos de variables
  PASO 2 - Calidad de datos: nulos, duplicados, valores anomalos
  PASO 3 - Analisis univariante: distribucion de la variable objetivo
  PASO 4 - Analisis bivariante: correlaciones / crosstabs con el estres
  PASO 5 - Hallazgos: sintesis de 3-5 conclusiones del dataset
================================================================================
"""
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# ── Configuracion global ──────────────────────────────────────────────────────
import os

BASE = r"C:\Users\nicop\OneDrive\Documentos\Maestria\INESDI & UNIE\BA & IA\TFM\Propuesta"
DATA = os.path.join(BASE, "Bloque 1") + os.sep
FIG  = os.path.join(BASE, "figuras1") + os.sep
os.makedirs(FIG, exist_ok=True)   # crea la carpeta de figuras1 si no existe

sns.set_theme(style="whitegrid", font_scale=0.95)
pd.set_option("display.width", 140)

SEP = "=" * 80

# ==============================================================================
# DATASET 1.1 — ACADEMIC STRESS (poushal02, n=140)
# ==============================================================================
print(SEP); print("DATASET 1.1 - ACADEMIC STRESS (poushal02)"); print(SEP)

# PASO 1: Carga e inventario -----------------------------------------------
df1 = pd.read_csv(DATA + "1 Student Academic Stress Real World Dataset/"
                         "academic Stress level - maintainance 1.csv")
df1.columns = [c.strip() for c in df1.columns]          # limpieza de espacios
TARGET1 = "Rate your academic stress index"
print(f"Dimensiones: {df1.shape[0]} filas x {df1.shape[1]} columnas")
print("Variables:", list(df1.columns))

# PASO 2: Calidad ------------------------------------------------------------
print("\nNulos por columna:\n", df1.isna().sum()[df1.isna().sum() > 0])
print("Duplicados:", df1.duplicated().sum())

# PASO 3: Univariante --------------------------------------------------------
dist1 = df1[TARGET1].value_counts().sort_index()
print("\nDistribucion del indice de estres (1-5):\n", dist1)
pct_alto = (df1[TARGET1] >= 4).mean() * 100
print(f"% de estudiantes en niveles 4-5: {pct_alto:.1f}%")

# PASO 4: Bivariante ---------------------------------------------------------
print("\nEstres medio por etapa academica:")
print(df1.groupby("Your Academic Stage")[TARGET1].agg(["mean", "count"]).round(2))
print("\nEstres medio segun presion de pares (peer pressure):")
print(df1.groupby("Peer pressure")[TARGET1].mean().round(2))

# Figura 1.1
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.countplot(data=df1, x=TARGET1, ax=axes[0], color="#1F5C99")
axes[0].set_title(f"Distribución del índice de estrés académico (n={len(df1)})")
axes[0].set_xlabel("Índice de estrés (1=bajo, 5=alto)")
axes[0].set_ylabel("Estudiantes")
sns.boxplot(data=df1, x="Your Academic Stage", y=TARGET1, ax=axes[1],
            hue="Your Academic Stage", palette="Blues", legend=False)
axes[1].set_title("Estrés por etapa académica")
axes[1].set_xlabel(""); axes[1].set_ylabel("Índice de estrés")
plt.tight_layout()
plt.savefig(FIG + "b1_1_academic_stress.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 1.1:
 - Mas del 60% de la muestra reporta estres en niveles 4-5.
 - La etapa de grado (undergraduate) concentra la mayoria de la muestra (100/140).
 - La presion de pares se asocia a mayor indice de estres.
 - Dataset pequeno (n=140): util para EDA descriptivo, limitado para modelar.
""")

# ==============================================================================
# DATASET 1.2 — ANXIETY / GAD-7 (petalme, n=13.464)
# ==============================================================================
print(SEP); print("DATASET 1.2 - ANXIETY GAD-7 (petalme)"); print(SEP)

# PASO 1: Carga (encoding latin1: el archivo no es UTF-8) --------------------
df2 = pd.read_csv(DATA + "1 Student anxiety dataset/anxiety.csv",
                  encoding="latin1")
print(f"Dimensiones: {df2.shape[0]} filas x {df2.shape[1]} columnas")
print("Escalas incluidas: GAD-7 (ansiedad), SWL (satisfaccion vital), "
      "SPIN (ansiedad social)")

# PASO 2: Calidad + subconjunto estudiantil ----------------------------------
nulos2 = df2[["GAD_T", "SWL_T", "SPIN_T", "Hours", "Age"]].isna().sum()
print("\nNulos en variables clave:\n", nulos2)
students = df2[df2["Work"].isin(
    ["Student at college / university", "Student at school"])].copy()
print(f"Subconjunto estudiantes (variable Work): n={len(students)}")

# PASO 3: Univariante --------------------------------------------------------
print("\nGAD_T (0-21) describe:\n", students["GAD_T"].describe().round(2))
# Umbrales clinicos GAD-7: 5=leve, 10=moderada, 15=severa
severidad = pd.cut(students["GAD_T"], bins=[-1, 4, 9, 14, 21],
                   labels=["Mínima", "Leve", "Moderada", "Severa"])
print("\nSeveridad clinica (umbrales GAD-7):\n",
      severidad.value_counts(normalize=True).round(3) * 100)

# PASO 4: Bivariante ---------------------------------------------------------
corr2 = students[["GAD_T", "SWL_T", "SPIN_T", "Hours", "Age"]].corr().round(2)
print("\nMatriz de correlaciones:\n", corr2)

# Figura 1.2
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.histplot(students["GAD_T"], bins=22, ax=axes[0], color="#1F5C99")
axes[0].set_title(f"Distribución GAD_T — ansiedad (estudiantes, n={len(students)})")
axes[0].set_xlabel("Puntuación GAD-7 (0-21)")
sns.scatterplot(data=students.sample(3000, random_state=42),
                x="SWL_T", y="GAD_T", alpha=0.2, ax=axes[1], color="#C44E52")
axes[1].set_title("Ansiedad vs Satisfacción con la vida (r = -0.40)")
axes[1].set_xlabel("SWL_T (satisfacción vital)"); axes[1].set_ylabel("GAD_T")
plt.tight_layout()
plt.savefig(FIG + "b1_2_anxiety_gad.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 1.2:
 - Correlacion GAD_T vs SWL_T = -0.40: mas ansiedad, menos satisfaccion vital.
 - Correlacion GAD_T vs SPIN_T = +0.44: la ansiedad general y la social co-ocurren.
 - Las horas de juego apenas correlacionan con la ansiedad (r=0.07).
 - Contexto original: jugadores online; el subset estudiantil (n=9.302) es valido
   para analizar ansiedad en jovenes, con esa salvedad documentada.
""")

# ==============================================================================
# DATASET 1.3 — MENTAL HEALTH SURVEY (abdullahashfaqvirk, n=87)
# ==============================================================================
print(SEP); print("DATASET 1.3 - MENTAL HEALTH SURVEY (abdullahashfaqvirk)"); print(SEP)

# PASO 1: Carga --------------------------------------------------------------
df3 = pd.read_csv(DATA + "1 Student Mental Health Survey/MentalHealthSurvey.csv")
print(f"Dimensiones: {df3.shape[0]} filas x {df3.shape[1]} columnas")

# PASO 2: Calidad ------------------------------------------------------------
print("Nulos totales:", df3.isna().sum().sum())
print("Nota: cgpa y average_sleep vienen como rangos de texto -> recodificar "
      "a ordinal antes del modelado (Objetivo 2).")

# PASO 3: Univariante --------------------------------------------------------
print("\nDistribucion depression (1-5):\n",
      df3["depression"].value_counts().sort_index())
print("Distribucion anxiety (1-5):\n",
      df3["anxiety"].value_counts().sort_index())

# PASO 4: Bivariante ---------------------------------------------------------
num3 = df3.select_dtypes("number")
corr3 = num3.corr()[["depression", "anxiety"]].round(2)
print("\nCorrelacion de cada variable con depression y anxiety:\n",
      corr3.sort_values("depression", ascending=False))

# Figura 1.3
plt.figure(figsize=(7.5, 6))
sns.heatmap(num3.corr(), annot=True, fmt=".2f", cmap="vlag", center=0,
            annot_kws={"size": 8})
plt.title(f"Matriz de correlaciones — Mental Health Survey (n={len(df3)})")
plt.tight_layout()
plt.savefig(FIG + "b1_3_mentalhealthsurvey_corr.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 1.3:
 - Aislamiento social: r=0.71 con depresion (el predictor mas fuerte del bloque).
 - Inseguridad ante el futuro: r=0.47 con depresion.
 - Carga academica: r=0.38 con depresion y ansiedad.
 - La satisfaccion academica actua como factor protector (r negativo).
 - n=87: correlaciones consistentes entre si a pesar del tamano reducido.
""")

# ==============================================================================
# DATASET 1.4 — STUDENT HABITS VS PERFORMANCE (abhi88377, n=1000)
# ==============================================================================
print(SEP); print("DATASET 1.4 - STUDENT HABITS VS PERFORMANCE (abhi88377)"); print(SEP)

# PASO 1: Carga --------------------------------------------------------------
df4 = pd.read_csv(DATA + "1 Student-Habits-vs-Academic-Performance/"
                         "student_habits_performance.csv")
print(f"Dimensiones: {df4.shape[0]} filas x {df4.shape[1]} columnas")

# PASO 2: Calidad ------------------------------------------------------------
print("Nulos:", dict(df4.isna().sum()[df4.isna().sum() > 0]))

# PASO 3: Univariante --------------------------------------------------------
print("\nmental_health_rating (1-10):\n",
      df4["mental_health_rating"].value_counts().sort_index())
print(">>> Distribucion casi uniforme: primer indicio de datos sinteticos.")

# PASO 4: Bivariante ---------------------------------------------------------
num4 = df4.select_dtypes("number")
print("\nCorrelacion con mental_health_rating:\n",
      num4.corr()["mental_health_rating"].round(2).sort_values())
print("\nCorrelacion con exam_score (validacion de coherencia interna):\n",
      num4.corr()["exam_score"].round(2).sort_values(ascending=False).head(4))

# Figura 1.4
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.histplot(df4["mental_health_rating"], bins=10, ax=axes[0], color="#55A868")
axes[0].set_title("mental_health_rating: distribución uniforme\n(indicio de datos sintéticos)")
sns.regplot(data=df4, x="study_hours_per_day", y="exam_score", ax=axes[1],
            scatter_kws={"alpha": 0.2}, line_kws={"color": "#C44E52"})
axes[1].set_title("Horas de estudio vs nota (r = 0.83)\núnica relación real del dataset")
plt.tight_layout()
plt.savefig(FIG + "b1_4_student_habits.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 1.4:
 - mental_health_rating NO correlaciona con ningun habito (r ~ 0.00 en todos).
 - La unica relacion fuerte es horas de estudio -> nota de examen (r=0.83).
 - CONCLUSION: dataset sintetico para la dimension de salud mental. Se usara
   para validar el pipeline tecnico del Objetivo 2, no para conclusiones.
 - 91 nulos (9.1%) en parental_education_level: imputar por moda o excluir.
""")

print(SEP)
print("EDA BLOQUE 1 COMPLETADO - 4 figuras generadas en", FIG)
print(SEP)

DATASET 1.1 - ACADEMIC STRESS (poushal02)
Dimensiones: 140 filas x 9 columnas
Variables: ['Timestamp', 'Your Academic Stage', 'Peer pressure', 'Academic pressure from your home', 'Study Environment', 'What coping strategy you use as a student?', 'Do you have any bad habits like smoking, drinking on a daily basis?', 'What would you rate the academic  competition in your student life', 'Rate your academic stress index']

Nulos por columna:
 Study Environment    1
dtype: int64
Duplicados: 0

Distribucion del indice de estres (1-5):
 Rate your academic stress index
1     6
2     9
3    36
4    56
5    33
Name: count, dtype: int64
% de estudiantes en niveles 4-5: 63.6%

Estres medio por etapa academica:
                     mean  count
Your Academic Stage             
high school          3.83     29
post-graduate        3.73     11
undergraduate        3.69    100

Estres medio segun presion de pares (peer pressure):
Peer pressure
1    2.62
2    3.29
3    3.75
4    4.06
5    4.57
Name: Rat

## Bloque 2 — Factores externos (hábitos digitales)

3 datasets: Social Media/Sleep, SMMH y Mental Health Lifestyle. SMMH es la fuente con señal real; los otros dos son sintéticos.

In [ ]:
# -*- coding: utf-8 -*-
"""
================================================================================
OBJETIVO 1 - ANALISIS EXPLORATORIO Y DESCRIPTIVO (EDA)
BLOQUE 2: FACTORES EXTERNOS (HABITOS DIGITALES)
================================================================================
Datasets analizados (3):
  2.1 social_media_sleep_stress_productivity_11000.csv  (Kaggle: jayjoshi37)
  2.2 smmh.csv                                           (Kaggle: souvikahmed071)
  2.3 Mental_Health_Lifestyle_Dataset.csv                (GitHub: datafit17)

Metodologia (misma plantilla de 5 pasos que el Bloque 1).
================================================================================
"""
# ── Configuracion global ──────────────────────────────────────────────────────

DATA = os.path.join(BASE, "Bloque 2") + os.sep

sns.set_theme(style="whitegrid", font_scale=0.95)
pd.set_option("display.width", 140)

SEP = "=" * 80

# ==============================================================================
# DATASET 2.1 — SOCIAL MEDIA / SLEEP / STRESS / PRODUCTIVITY (jayjoshi37, n=11000)
# ==============================================================================
print(SEP); print("DATASET 2.1 - SOCIAL MEDIA SLEEP STRESS (jayjoshi37)"); print(SEP)

# PASO 1: Carga --------------------------------------------------------------
df1 = pd.read_csv(DATA + "2 Social Media Usage vs Sleep, Stress & Productivity/"
                         "social_media_sleep_stress_productivity_11000.csv")
print(f"Dimensiones: {df1.shape[0]} filas x {df1.shape[1]} columnas")

# PASO 2: Calidad ------------------------------------------------------------
print("Nulos totales:", df1.isna().sum().sum())

# PASO 3: Univariante --------------------------------------------------------
print("\nDistribucion stress_level:\n", df1["stress_level"].value_counts())
print(">>> Balance exacto ~33% por clase: primer indicio de datos sinteticos.")
print("\nDistribucion por plataforma:\n", df1["platform"].value_counts())

# PASO 4: Bivariante (codificacion ordinal del estres) -----------------------
df1["stress_ord"] = df1["stress_level"].map({"Low": 0, "Medium": 1, "High": 2})
num1 = df1.select_dtypes("number").drop(columns=["user_id"])
print("\nCorrelacion con stress_ord:\n",
      num1.corr()["stress_ord"].round(3).sort_values())
print("\nMedias por nivel de estres (sin diferencias entre grupos):")
print(df1.groupby("stress_level")[["social_media_hours", "sleep_hours",
      "productivity_score"]].mean().round(2))

# Figura 2.1
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.countplot(data=df1, x="stress_level", order=["Low", "Medium", "High"],
              ax=axes[0], color="#1F5C99")
axes[0].set_title(f"stress_level perfectamente balanceado (n={len(df1)})\nindicio de datos sintéticos")
sns.heatmap(num1.corr(), annot=True, fmt=".2f", cmap="vlag", center=0,
            ax=axes[1], annot_kws={"size": 8})
axes[1].set_title("Correlaciones ≈ 0 en todas las variables")
plt.tight_layout()
plt.savefig(FIG + "b2_1_social_media_sleep.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 2.1:
 - Todas las correlaciones con el estres son <=0.02: sin senal real.
 - Distribuciones balanceadas artificialmente (clases y plataformas).
 - CONCLUSION: dataset sintetico. Uso: validacion tecnica de pipeline y
   dashboards de volumen, no conclusiones sustantivas.
""")

# ==============================================================================
# DATASET 2.2 — SOCIAL MEDIA AND MENTAL HEALTH / SMMH (souvikahmed071, n=481)
# ==============================================================================
print(SEP); print("DATASET 2.2 - SMMH (souvikahmed071)"); print(SEP)

# PASO 1: Carga --------------------------------------------------------------
df2 = pd.read_csv(DATA + "2 Social Media and Mental Health/smmh.csv")
cols = list(df2.columns)
print(f"Dimensiones: {df2.shape[0]} filas x {df2.shape[1]} columnas")
print("Encuesta con escalas Likert 1-5 sobre uso de redes y bienestar.")

# PASO 2: Calidad ------------------------------------------------------------
print("Nulos totales:", df2.isna().sum().sum(),
      "(30 en columna de organizacion, no afecta el analisis)")

# PASO 3: Univariante --------------------------------------------------------
target2 = cols[18]   # "how often do you feel depressed or down?"
print(f"\nVariable objetivo: '{target2}'")
print(df2[target2].value_counts().sort_index())
print("\nTiempo diario en redes:\n", df2[cols[8]].value_counts())

# PASO 4: Bivariante ---------------------------------------------------------
likert_idx = [9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20]
likert_cols = [cols[i] for i in likert_idx]
num2 = df2[likert_cols].apply(pd.to_numeric, errors="coerce")
corr2 = num2.corr()[target2].round(2).sort_values(ascending=False)
print("\nCorrelaciones con sentirse deprimido/a:\n", corr2)

# Figura 2.2 (etiquetas cortas para legibilidad)
short = {
    cols[9]:  "Uso sin propósito",
    cols[10]: "Distracción por redes",
    cols[11]: "Inquietud sin redes",
    cols[12]: "Distracción fácil",
    cols[13]: "Preocupación excesiva",
    cols[14]: "Dificultad concentración",
    cols[15]: "Comparación social",
    cols[17]: "Búsqueda de validación",
    cols[19]: "Fluctuación de interés",
    cols[20]: "Problemas de sueño",
}
plot_data = corr2.drop(target2).rename(index=short)
plt.figure(figsize=(8, 5))
plot_data.sort_values().plot(kind="barh", color="#C44E52")
plt.title(f"Correlación con «sentirse deprimido/a» (n={len(df2)})\nSMMH — souvikahmed071")
plt.xlabel("Pearson r")
plt.tight_layout()
plt.savefig(FIG + "b2_2_smmh_corr.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 2.2:
 - Preocupacion excesiva: r=0.59 con sentirse deprimido/a (mayor del bloque).
 - Dificultad de concentracion: r=0.51; fluctuacion de interes: r=0.49.
 - Comparacion social: r=0.42; problemas de sueno: r=0.37.
 - CONCLUSION: no es el tiempo total en redes sino el patron de uso ansioso
   y comparativo lo que se asocia al malestar. Dataset mas valioso del bloque.
""")

# ==============================================================================
# DATASET 2.3 — MENTAL HEALTH LIFESTYLE (datafit17, n=3000)
# ==============================================================================
print(SEP); print("DATASET 2.3 - MENTAL HEALTH LIFESTYLE (datafit17)"); print(SEP)

# PASO 1: Carga --------------------------------------------------------------
df3 = pd.read_csv(DATA + "2 Mental_Health_Lifestyle GITHUB/"
                         "Mental_Health_Lifestyle_Dataset.csv")
print(f"Dimensiones: {df3.shape[0]} filas x {df3.shape[1]} columnas")

# PASO 2: Calidad (recodificacion de NaN semanticos) --------------------------
n_nan = df3["Mental Health Condition"].isna().sum()
df3["Mental Health Condition"] = df3["Mental Health Condition"].fillna("Ninguna")
print(f"NaN en Mental Health Condition: {n_nan} ({n_nan/len(df3)*100:.1f}%) "
      "-> recodificados como 'Ninguna' (ausencia de diagnostico, no dato faltante)")

# PASO 3: Univariante --------------------------------------------------------
print("\nDistribucion Stress Level:\n", df3["Stress Level"].value_counts())
print(">>> De nuevo balance casi exacto: indicio de datos sinteticos.")

# PASO 4: Bivariante ---------------------------------------------------------
df3["stress_ord"] = df3["Stress Level"].map({"Low": 0, "Moderate": 1, "High": 2})
num3 = df3.select_dtypes("number")
print("\nCorrelacion con stress_ord:\n",
      num3.corr()["stress_ord"].round(3).sort_values())

# Figura 2.3
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
orden = df3["Mental Health Condition"].value_counts().index
sns.countplot(data=df3, y="Mental Health Condition", order=orden,
              ax=axes[0], color="#8172B2")
axes[0].set_title(f"Condición de salud mental (n={len(df3)})")
sns.boxplot(data=df3, x="Stress Level", y="Sleep Hours",
            order=["Low", "Moderate", "High"], ax=axes[1],
            hue="Stress Level", palette="Blues", legend=False)
axes[1].set_title("Horas de sueño por nivel de estrés\n(sin diferencias entre grupos)")
plt.tight_layout()
plt.savefig(FIG + "b2_3_mental_health_lifestyle.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 2.3:
 - Correlaciones ~0.00-0.02 entre estres y todas las variables de estilo de vida.
 - Distribucion de estres artificialmente balanceada (1008/990/1002).
 - CONCLUSION: dataset sintetico. Mismo tratamiento que 2.1: valor tecnico
   (pipeline, dashboards multi-pais), no valor inferencial.
""")

print(SEP)
print("EDA BLOQUE 2 COMPLETADO - 3 figuras generadas en", FIG)
print(SEP)

DATASET 2.1 - SOCIAL MEDIA SLEEP STRESS (jayjoshi37)
Dimensiones: 11000 filas x 10 columnas
Nulos totales: 0

Distribucion stress_level:
 stress_level
Low       3715
Medium    3681
High      3604
Name: count, dtype: int64
>>> Balance exacto ~33% por clase: primer indicio de datos sinteticos.

Distribucion por plataforma:
 platform
LinkedIn     2239
Snapchat     2235
YouTube      2200
Twitter      2183
Instagram    2143
Name: count, dtype: int64

Correlacion con stress_ord:
 age                       -0.021
exercise_minutes          -0.012
daily_screen_time_hours   -0.010
social_media_hours        -0.008
productivity_score        -0.004
sleep_hours                0.000
study_work_hours           0.005
stress_ord                 1.000
Name: stress_ord, dtype: float64

Medias por nivel de estres (sin diferencias entre grupos):
              social_media_hours  sleep_hours  productivity_score
stress_level                                                     
High                        4.21

## Bloque 3 — Percepción emocional (IA / NLP)

2 datasets: Dreaddit (corpus de texto real de Reddit, la fuente de NLP) y Depression Student (tabular).

In [ ]:
# -*- coding: utf-8 -*-
"""
================================================================================
OBJETIVO 1 - ANALISIS EXPLORATORIO Y DESCRIPTIVO (EDA)
BLOQUE 3: PERCEPCION EMOCIONAL (IA / NLP)
================================================================================
Datasets analizados (2):
  3.1 dreaddit-train.csv + dreaddit-test.csv     (Kaggle: payaldhokane)
      -> corpus textual real de Reddit con etiqueta de estres. Fuente NLP.
  3.2 Depression+Student+Dataset.csv             (GitHub: prajwalg26)
      -> tabular; complementa el perfil de malestar estudiantil. NO es NLP.

Para el corpus textual, el EDA incluye pasos especificos de texto:
  PASO 3-T: longitud de posts, balance de clases, distribucion por subreddit
  PASO 4-T: senal lexica (sentimiento, categorias LIWC) por clase
================================================================================
"""
DATA = os.path.join(BASE, "Bloque 3") + os.sep

sns.set_theme(style="whitegrid", font_scale=0.95)
pd.set_option("display.width", 140)

SEP = "=" * 80

# ==============================================================================
# DATASET 3.1 — DREADDIT (payaldhokane, train n=2838 / test n=715)
# ==============================================================================
print(SEP); print("DATASET 3.1 - DREADDIT (payaldhokane)"); print(SEP)

# PASO 1: Carga --------------------------------------------------------------
tr = pd.read_csv(DATA + "3 Stress Analysis in Social Media Dataset/dreaddit-train.csv")
te = pd.read_csv(DATA + "3 Stress Analysis in Social Media Dataset/dreaddit-test.csv")
print(f"Train: {tr.shape} | Test: {te.shape}")
print("Columnas clave: text (post), label (0/1), subreddit, sentiment, "
      "107 categorias LIWC, indices DAL y legibilidad (116 columnas total)")

# PASO 2: Calidad ------------------------------------------------------------
print("Nulos train:", tr.isna().sum().sum(), "| Nulos test:", te.isna().sum().sum())
print("Particion train/test ya definida por los autores: se respetara en el "
      "Objetivo 2 para comparabilidad con la literatura.")

# PASO 3-T: EDA textual - balance, longitud, subreddits ------------------------
tr["text_len"] = tr["text"].str.len()
tr["n_words"] = tr["text"].str.split().str.len()
print("\nBalance de clases:\n", tr["label"].value_counts())
print("\nLongitud (palabras) por clase:\n",
      tr.groupby("label")["n_words"].describe()[["mean", "50%", "max"]].round(1))
print("\nPosts por subreddit:\n", tr["subreddit"].value_counts())

# PASO 4-T: Senal lexica por clase --------------------------------------------
print("\nSentimiento medio por clase:\n",
      tr.groupby("label")["sentiment"].mean().round(3))
liwc_vars = ["lex_liwc_anx", "lex_liwc_negemo", "lex_liwc_posemo", "lex_liwc_i"]
print("\nCategorias LIWC por clase (medias):")
print(tr.groupby("label")[liwc_vars].mean().round(2))

# Figura 3.1a - balance, sentimiento, LIWC ansiedad
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
sns.countplot(data=tr, x="label", ax=axes[0], color="#937860")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["No estrés (0)", "Estrés (1)"])
axes[0].set_title(f"Balance de clases (train, n={len(tr)})")
sns.boxplot(data=tr, x="label", y="sentiment", ax=axes[1],
            hue="label", palette=["#55A868", "#C44E52"], legend=False)
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(["No estrés", "Estrés"])
axes[1].set_title("Sentimiento por clase")
sns.barplot(data=tr, x="label", y="lex_liwc_anx", ax=axes[2],
            hue="label", palette=["#55A868", "#C44E52"], legend=False)
axes[2].set_xticks([0, 1]); axes[2].set_xticklabels(["No estrés", "Estrés"])
axes[2].set_title("Densidad de palabras de ansiedad (LIWC)\n2.5× mayor en la clase estrés")
plt.tight_layout()
plt.savefig(FIG + "b3_1_dreaddit_senales.png", dpi=130)
plt.close()

# Figura 3.1b - longitud y subreddits
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.histplot(data=tr, x="n_words", hue="label", bins=40, ax=axes[0],
             palette=["#55A868", "#C44E52"])
axes[0].set_title("Longitud de los posts (palabras) por clase")
axes[0].set_xlabel("Número de palabras")
tr["subreddit"].value_counts().sort_values().plot(kind="barh", ax=axes[1], color="#64B5CD")
axes[1].set_title("Posts por subreddit de origen")
plt.tight_layout()
plt.savefig(FIG + "b3_1_dreaddit_texto.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 3.1:
 - Clases casi balanceadas (1488 estres / 1350 no-estres): optimo para clasificar.
 - Sentimiento: +0.103 (no estres) vs -0.016 (estres).
 - Palabras de ansiedad LIWC: 1.28 vs 0.51 (x2.5 en clase estres).
 - Emocion negativa LIWC tambien discrimina; emocion positiva es mayor en no-estres.
 - Subreddits dominantes: ptsd, relationships, anxiety -> el estres capturado es
   relacional/clinico, no exclusivamente academico (considerar al generalizar).
 - 116 features precalculadas permiten comparar en el Objetivo 2 un modelo LIWC
   vs un modelo TF-IDF/embeddings propio.
""")

# ==============================================================================
# DATASET 3.2 — DEPRESSION STUDENT DATASET (prajwalg26, n=502)
# ==============================================================================
print(SEP); print("DATASET 3.2 - DEPRESSION STUDENT (prajwalg26)"); print(SEP)

# PASO 1: Carga --------------------------------------------------------------
df2 = pd.read_csv(DATA + "3 Student_Mental_Health_Analysis GITHUB/"
                         "Depression+Student+Dataset.csv")
print(f"Dimensiones: {df2.shape[0]} filas x {df2.shape[1]} columnas")
print("NOTA METODOLOGICA: dataset tabular (sin texto). Complementa el bloque con "
      "un perfil estructurado del malestar estudiantil; el NLP recae en Dreaddit.")

# PASO 2: Calidad ------------------------------------------------------------
print("Nulos totales:", df2.isna().sum().sum(),
      "| Duplicados:", df2.duplicated().sum())

# PASO 3: Univariante --------------------------------------------------------
print("\nBalance del target Depression:\n", df2["Depression"].value_counts())

# PASO 4: Bivariante ---------------------------------------------------------
ct = pd.crosstab(df2["Have you ever had suicidal thoughts ?"],
                 df2["Depression"], normalize="index") * 100
print("\n% Depresion segun pensamientos suicidas:\n", ct.round(1))
print("\nMedias por grupo de depresion:")
print(df2.groupby("Depression")[["Academic Pressure", "Study Satisfaction",
      "Financial Stress", "Study Hours"]].mean().round(2))

# Figura 3.2
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
ct["Yes"].plot(kind="bar", ax=axes[0], color=["#55A868", "#C44E52"])
axes[0].set_title("% con depresión según pensamientos suicidas")
axes[0].set_xlabel("¿Ha tenido pensamientos suicidas?")
axes[0].set_ylabel("% con depresión"); axes[0].tick_params(axis="x", rotation=0)
sns.boxplot(data=df2, x="Depression", y="Academic Pressure", ax=axes[1],
            hue="Depression", palette=["#55A868", "#C44E52"], legend=False)
axes[1].set_title("Presión académica según depresión")
plt.tight_layout()
plt.savefig(FIG + "b3_2_depression_student.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 3.2:
 - 73% de quienes reportan pensamientos suicidas presentan depresion (vs 26%).
 - Presion academica: 3.66 (con depresion) vs 2.34 (sin) -> +57%.
 - Estres financiero tambien es mayor en el grupo con depresion.
 - Target balanceado (252/250): apto para clasificacion en el Objetivo 2.
""")

print(SEP)
print("EDA BLOQUE 3 COMPLETADO - 3 figuras generadas en", FIG)
print(SEP)

DATASET 3.1 - DREADDIT (payaldhokane)
Train: (2838, 116) | Test: (715, 116)
Columnas clave: text (post), label (0/1), subreddit, sentiment, 107 categorias LIWC, indices DAL y legibilidad (116 columnas total)
Nulos train: 0 | Nulos test: 0
Particion train/test ya definida por los autores: se respetara en el Objetivo 2 para comparabilidad con la literatura.

Balance de clases:
 label
1    1488
0    1350
Name: count, dtype: int64

Longitud (palabras) por clase:
 

C:\Users\nicop\AppData\Local\Temp\ipykernel_8980\3604628787.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tr["text_len"] = tr["text"].str.len()
C:\Users\nicop\AppData\Local\Temp\ipykernel_8980\3604628787.py:44: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tr["n_words"] = tr["text"].str.split().str.len()


       mean   50%    max
label                   
0      82.2  78.0  255.0
1      88.8  83.0  310.0

Posts por subreddit:
 subreddit
ptsd                584
relationships       552
anxiety             503
domesticviolence    316
assistance          289
survivorsofabuse    245
homeless            168
almosthomeless       80
stress               64
food_pantry          37
Name: count, dtype: int64

Sentimiento medio por clase:
 label
0    0.103
1   -0.016
Name: sentiment, dtype: float64

Categorias LIWC por clase (medias):
       lex_liwc_anx  lex_liwc_negemo  lex_liwc_posemo  lex_liwc_i
label                                                            
0              0.51             2.07             3.42        7.11
1              1.28             4.42             2.04       10.84

HALLAZGOS 3.1:
 - Clases casi balanceadas (1488 estres / 1350 no-estres): optimo para clasificar.
 - Sentimiento: +0.103 (no estres) vs -0.016 (estres).
 - Palabras de ansiedad LIWC: 1.28 vs 0.51 (x2.5 en cla

## Bloque 4 — Validación psicológica

2 datasets: DASS-42 (escala clínica validada, con su codebook) y Mental Health Dataset (la mayor fuente, con la brecha de tratamiento).

In [ ]:
# -*- coding: utf-8 -*-
"""
================================================================================
OBJETIVO 1 - ANALISIS EXPLORATORIO Y DESCRIPTIVO (EDA)
BLOQUE 4: VALIDACION PSICOLOGICA
================================================================================
Datasets analizados (2):
  4.1 data.csv (DASS-42)                    (Kaggle: lucasgreenwell)
      + codebook oficial: "Depression Anxiety Stress Scales Responses.txt"
  4.2 Mental Health dataset1.csv            (Kaggle: chanchal24 / input)

El DASS-42 es una escala clinicamente validada. La clave de subescalas y los
codigos demograficos provienen del codebook oficial incluido en el ZIP.
================================================================================
"""
DATA = os.path.join(BASE, "Bloque 4") + os.sep

sns.set_theme(style="whitegrid", font_scale=0.95)
pd.set_option("display.width", 140)

SEP = "=" * 80

# ==============================================================================
# DATASET 4.1 — DASS-42 (lucasgreenwell, n=39.775)
# ==============================================================================
print(SEP); print("DATASET 4.1 - DASS-42 (lucasgreenwell)"); print(SEP)

# PASO 1: Carga (separador TAB, no coma) --------------------------------------
df = pd.read_csv(DATA + "4 Depression Anxiety Stress Scales Responses/data.csv",
                 sep="\t")
print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
print("42 items (respuesta A, tiempo E, posicion I) + TIPI + demograficos")

# PASO 2: Calidad - limpieza de edades invalidas -------------------------------
# El codebook confirma que 'age' es respuesta libre: hay valores tipo 1998 (ano)
n_before = len(df)
df = df[(df["age"] >= 13) & (df["age"] <= 90)].copy()
print(f"Filtrado de edad a rango 13-90: {n_before} -> {len(df)} "
      f"({n_before - len(df)} registros eliminados)")

# PASO 3: Calculo de subescalas (clave oficial DASS-42 del codebook) ----------
dep_items = [3, 5, 10, 13, 16, 17, 21, 24, 26, 31, 34, 37, 38, 42]
anx_items = [2, 4, 7, 9, 15, 19, 20, 23, 25, 28, 30, 36, 40, 41]
str_items = [1, 6, 8, 11, 12, 14, 18, 22, 27, 29, 32, 33, 35, 39]

def subescala(items):
    """Suma las respuestas (QnA) de los items de una subescala. Rango 14-56."""
    return df[[f"Q{i}A" for i in items]].sum(axis=1)

df["dep_score"] = subescala(dep_items)
df["anx_score"] = subescala(anx_items)
df["str_score"] = subescala(str_items)

# Subconjunto proxy de poblacion estudiantil: 18-25 anos
stu = df[(df["age"] >= 18) & (df["age"] <= 25)].copy()
print(f"\nSubconjunto proxy estudiantil (18-25 anos): n={len(stu)}")
print("\nEstadisticos de las subescalas (subset 18-25):")
print(stu[["dep_score", "anx_score", "str_score"]].describe().round(1).loc[
      ["mean", "std", "50%"]])

# PASO 4: Desagregacion demografica (codigos del codebook oficial) ------------
stu["genero"] = stu["gender"].map({1: "Hombre", 2: "Mujer", 3: "Otro"})
stu["educacion"] = stu["education"].map({1: "Menos que secundaria",
                                          2: "Secundaria",
                                          3: "Grado universitario",
                                          4: "Posgrado"})
print("\nSubescalas medias por genero:\n",
      stu.groupby("genero")[["dep_score", "anx_score", "str_score"]].mean().round(1))
print("\nSubescalas medias por nivel educativo:\n",
      stu.groupby("educacion")[["dep_score", "anx_score", "str_score"]].mean().round(1))

# Figura 4.1a - distribuciones de las 3 subescalas
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
nombres = {"dep_score": "Depresión", "anx_score": "Ansiedad", "str_score": "Estrés"}
for ax, (col, name) in zip(axes, nombres.items()):
    sns.histplot(stu[col], bins=21, ax=ax, color="#1F5C99")
    ax.axvline(stu[col].mean(), color="#C44E52", ls="--", lw=1.5,
               label=f"media={stu[col].mean():.1f}")
    ax.set_title(f"DASS — {name}"); ax.set_xlabel("Puntuación (14-56)")
    ax.legend()
fig.suptitle(f"Subescalas DASS-42, subset 18-25 años (n={len(stu)})", y=1.03)
plt.tight_layout()
plt.savefig(FIG + "b4_1_dass_subescalas.png", dpi=130, bbox_inches="tight")
plt.close()

# Figura 4.1b - desagregacion demografica
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
stu.groupby("genero")[["dep_score", "anx_score", "str_score"]].mean() \
   .rename(columns=nombres) \
   .plot(kind="bar", ax=axes[0], color=["#1F5C99", "#DD8452", "#55A868"])
axes[0].set_title("DASS medio por género"); axes[0].tick_params(axis="x", rotation=0)
axes[0].set_ylabel("Puntuación media")
orden_edu = ["Menos que secundaria", "Secundaria", "Grado universitario", "Posgrado"]
stu.groupby("educacion")[["dep_score", "anx_score", "str_score"]].mean() \
   .reindex(orden_edu).rename(columns=nombres) \
   .plot(kind="bar", ax=axes[1], color=["#1F5C99", "#DD8452", "#55A868"])
axes[1].set_title("DASS medio por nivel educativo")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig(FIG + "b4_1_dass_demograficos.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 4.1:
 - Subset 18-25 (n=23.031): medias Dep=35.0, Anx=30.4, Str=35.2 (escala 14-56).
 - Mujeres reportan mas ansiedad (31.0) que hombres (28.1); el grupo 'Otro'
   presenta el mayor malestar en las tres subescalas.
 - A menor nivel educativo completado, mayor puntuacion en las 3 subescalas.
 - Referencia normativa robusta para contrastar los niveles de estres
   autoreportados en los Bloques 1 y 2 (sin fusionar datos).
""")

# ==============================================================================
# DATASET 4.2 — MENTAL HEALTH DATASET (chanchal24, n=261.328)
# ==============================================================================
print(SEP); print("DATASET 4.2 - MENTAL HEALTH DATASET (chanchal24)"); print(SEP)

# PASO 1: Carga --------------------------------------------------------------
df2 = pd.read_csv(DATA + "4 Mental Health Dataset/Mental Health dataset1.csv")
print(f"Dimensiones: {df2.shape[0]} filas x {df2.shape[1]} columnas")

# PASO 2: Calidad - columna duplicada y nulos ---------------------------------
if (df2["SocialWeakness"] == df2["SocialWeakness.1"]).all():
    df2 = df2.drop(columns=["SocialWeakness.1"])
    print("Columna 'SocialWeakness.1' eliminada (duplicado exacto).")
print("Nulos:", dict(df2.isna().sum()[df2.isna().sum() > 0]))

# PASO 3: Filtro a poblacion estudiantil --------------------------------------
stu2 = df2[df2["Occupation"] == "Student"].copy()
print(f"\nSubconjunto Occupation=='Student': n={len(stu2)}")
print("\nDistribucion IncreasingStress (estudiantes):\n",
      stu2["IncreasingStress"].value_counts())

# PASO 4: Bivariante - estres creciente vs busqueda de tratamiento ------------
ct = pd.crosstab(stu2["IncreasingStress"], stu2["Treatment"],
                 normalize="index") * 100
print("\n% que busca tratamiento segun estres creciente:\n", ct.round(1))
ct2 = pd.crosstab(stu2["FamilyHistory"], stu2["Treatment"], normalize="index") * 100
print("\n% que busca tratamiento segun antecedentes familiares:\n", ct2.round(1))

# Figura 4.2
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
ct.plot(kind="bar", stacked=True, ax=axes[0], color=["#CCCCCC", "#1F5C99"])
axes[0].set_title(f"Tratamiento según estrés creciente\n(estudiantes, n={len(stu2)})")
axes[0].set_ylabel("%"); axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(title="Tratamiento", loc="lower right")
ct2.plot(kind="bar", stacked=True, ax=axes[1], color=["#CCCCCC", "#C44E52"])
axes[1].set_title("Tratamiento según antecedentes familiares")
axes[1].set_ylabel("%"); axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="Tratamiento", loc="lower right")
plt.tight_layout()
plt.savefig(FIG + "b4_2_chanchal_tratamiento.png", dpi=130)
plt.close()

# PASO 5: Hallazgos ----------------------------------------------------------
print("""
HALLAZGOS 4.2:
 - La busqueda de tratamiento apenas varia con el estres creciente (48-51%):
   posible brecha de acceso o estigma en poblacion estudiantil.
 - Los antecedentes familiares SI elevan la busqueda de tratamiento.
 - Dataset masivo (54.207 estudiantes): apto para segmentaciones en dashboards.
""")

print(SEP)
print("EDA BLOQUE 4 COMPLETADO - 3 figuras generadas en", FIG)
print(SEP)

DATASET 4.1 - DASS-42 (lucasgreenwell)
Dimensiones: 39775 filas x 172 columnas
42 items (respuesta A, tiempo E, posicion I) + TIPI + demograficos
Filtrado de edad a rango 13-90: 39775 -> 39767 (8 registros eliminados)

Subconjunto proxy estudiantil (18-25 anos): n=23039

Estadisticos de las subescalas (subset 18-25):
      dep_score  anx_score  str_score
mean       35.0       30.4       35.2
std        12.0        9.9       10.3
50%        35.0       29.0       35.0

Subescalas medias por genero:
         dep_score  anx_score  str_score
genero                                 
Hombre       34.4       28.1       32.8
Mujer        35.1       31.0       35.7
Otro         40.2       33.0       37.8

Subescalas medias por nivel educativo:
                       dep_score  anx_score  str_score
educacion                                            
Grado universitario        34.2       30.1       34.8
Menos que secundaria       39.1       32.9       37.5
Posgrado                   34.1       29

---
# PARTE B · Entrega 2 — ETL y KPIs
## B.1 — Ingesta de los 12 CSV a MySQL

Lee los CSV, calcula en cada fila sus **claves conformes** (género, rango de edad, nivel educativo y nivel de estrés) y los inserta en las tablas del modelo v2 de la base `tfm_estres_estudiantil_2`.

**Cómo se protege de duplicados:** antes de cargar cuenta las filas de las 11 tablas.
- Si todas están vacías → carga los datos.
- Si todas ya tienen datos → **omite la carga** y sigue (así *Run All* nunca duplica).
- Si unas sí y otras no (carga a medias) → se detiene y explica cómo recargar limpio.

La tabla `b4_dass_item` (≈1,67 M de filas) no se carga por defecto (`CARGAR_DASS_ITEM = False`); no la necesitan los dashboards.

### Paso 0 — Instalar conectores (`sqlalchemy` y `pymysql`)

In [ ]:
# Instalar los conectores (solo la primera vez; si ya los tienes, no hace nada)
%pip install sqlalchemy pymysql --quiet
print("Conectores listos. Reinicia el kernel si es la primera instalacion.")

### Paso 1 — Configuración y conexión
Ajusta `DB_PASS` si tu MySQL tiene contraseña. `CARGAR_DASS_ITEM` queda en `False`: esa tabla son ~1,67 millones de filas y no la necesitan los dashboards.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

# ── Configuracion global ──────────────────────────────────────────────────────
BASE = r"C:\Users\nicop\OneDrive\Documentos\Maestria\INESDI & UNIE\BA & IA\TFM\Propuesta"

DB_USER, DB_PASS = "root", ""            # credenciales XAMPP por defecto
DB_HOST, DB_NAME = "localhost", "tfm_estres_estudiantil_2"
CARGAR_DASS_ITEM = False                  # ver nota de cabecera

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}?charset=utf8mb4")

# ── Claves de las dimensiones conformes (segun INSERTs del modelo v2) ─────────
ID_GENERO = {"NR": 0, "M": 1, "F": 2, "O": 3}
ID_RANGO  = {"No reportado": 0, "NR": 0, "<18": 1, "18-25": 2,
             "26-35": 3, "36-50": 4, ">50": 5}
ID_EDU    = {"NR": 0, "No reportado": 0, "Secundaria o menos": 1,
             "Grado universitario": 2, "Posgrado": 3}
ID_ESTRES = {"No aplica": 0, "Bajo": 1, "Moderado": 2, "Alto": 3}
ID_SUBRED = {n: i for i, n in enumerate(
    ["ptsd", "relationships", "anxiety", "domesticviolence", "assistance",
     "survivorsofabuse", "homeless", "almosthomeless", "stress", "food_pantry"], 1)}

### Paso 1 — Comprobación previa (evita duplicados)
Decide si hay que cargar (`OMITIR_CARGA = False`) o si los datos ya están (`OMITIR_CARGA = True`).

In [ ]:
# VERIFICACION DE SEGURIDAD: decide si se carga o se omite la carga (nunca duplica)
tablas_hechos = ["b1_academic_stress", "b1_anxiety", "b1_mental_health_survey",
                 "b1_student_habits", "b2_social_media_sleep", "b2_smmh",
                 "b2_lifestyle", "b3_dreaddit_post", "b3_depression_student",
                 "b4_dass_respondente", "b4_mental_health"]

with engine.connect() as con:
    print("Conexion OK ->", DB_NAME, "\n")
    ocupadas, vacias = [], []
    for t in tablas_hechos:
        n = con.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar()
        print(f"  {t:<26} {n:>8} filas")
        (ocupadas if n > 0 else vacias).append(t)

if ocupadas and vacias:
    raise RuntimeError(
        "\n*** CARGA A MEDIAS: tienen datos " + ", ".join(ocupadas) +
        "\n*** y estan vacias " + ", ".join(vacias) +
        "\n*** Para recargar limpio: reimporta modelo_datos_mysql_v2.sql y "
        "esquema_estrella_powerbi.sql en phpMyAdmin y vuelve a ejecutar el notebook.")

OMITIR_CARGA = bool(ocupadas)
if OMITIR_CARGA:
    print("\nTodas las tablas YA tienen datos: se OMITE la carga (no se duplica nada).")
else:
    print("\nTodas las tablas vacias: se procede con la carga.")

### Paso 3 — Funciones de armonización
Mismos criterios que el notebook 06: traducen género, edad, educación y la variable objetivo nativa a las claves de las dimensiones conformes.

In [ ]:
# ── Mapeadores (identicos al script 06) ───────────────────────────────────────
def map_genero(v):
    if pd.isna(v): return "NR"
    s = str(v).strip().lower()
    if s in ("male", "m", "hombre", "1", "1.0"):   return "M"
    if s in ("female", "f", "mujer", "2", "2.0"):  return "F"
    if s in ("", "nan", "prefer not to say"):      return "NR"
    return "O"

def map_rango_edad(v):
    try: e = float(v)
    except (TypeError, ValueError): return "NR"
    if pd.isna(e) or e < 13 or e > 90: return "NR"
    if e < 18:  return "<18"
    if e <= 25: return "18-25"
    if e <= 35: return "26-35"
    if e <= 50: return "36-50"
    return ">50"

def likert_a_nivel(v):
    try: x = float(v)
    except (TypeError, ValueError): return "No aplica"
    if x <= 2: return "Bajo"
    if x <= 3: return "Moderado"
    return "Alto"

def edad_o_none(v):
    """Edad valida 13-90 o None (para columnas TINYINT con CHECK)."""
    try: e = float(v)
    except (TypeError, ValueError): return None
    return int(e) if 13 <= e <= 90 else None

def conformes(df, genero=None, edad=None, edu=None, estres=None):
    """Devuelve las 4 columnas de claves conformes para el DataFrame dado."""
    out = pd.DataFrame(index=df.index)
    out["id_genero"] = (genero.map(map_genero) if genero is not None
                        else "NR")
    out["id_genero"] = out["id_genero"].map(ID_GENERO)
    out["id_rango_edad"] = (edad.map(map_rango_edad) if edad is not None
                            else "NR")
    out["id_rango_edad"] = out["id_rango_edad"].map(ID_RANGO)
    out["id_nivel_educativo"] = (edu if edu is not None else "NR")
    out["id_nivel_educativo"] = out["id_nivel_educativo"].fillna("NR").map(ID_EDU).fillna(0).astype(int)
    out["id_nivel_estres"] = estres.map(ID_ESTRES).fillna(0).astype(int)
    return out

def cargar(nombre_tabla, df):
    df.to_sql(nombre_tabla, engine, if_exists="append", index=False,
              chunksize=2000)
    print(f"  {nombre_tabla:<26} {len(df):>7} filas cargadas")

print("=" * 80)
print("CARGA DE DATOS - modelo v2 con dimensiones conformes")
print("=" * 80)

### Carga por dataset
Cada celda carga un dataset y muestra las filas insertadas. Si los datos ya estaban cargados, no hace nada.

**B1.1 Academic Stress** (140 filas). Sin género ni edad en el origen → claves conformes a *No reportado*; la etapa académica se aprovecha como nivel educativo.

In [ ]:
if OMITIR_CARGA:
    print('  (carga omitida: datos ya presentes)')
else:
    df = pd.read_csv(os.path.join(BASE, "Bloque 1",
        "1 Student Academic Stress Real World Dataset",
        "academic Stress level - maintainance 1.csv"))
    df.columns = [c.strip() for c in df.columns]
    T = "Rate your academic stress index"
    edu_stage = df["Your Academic Stage"].map({
        "undergraduate": "Grado universitario", "post-graduate": "Posgrado",
        "high school": "Secundaria o menos"})
    out = conformes(df, edu=edu_stage, estres=df[T].map(likert_a_nivel))
    out["id_dataset"] = 1
    out["ts_respuesta"] = df["Timestamp"].astype(str).str[:30]
    out["etapa_academica"] = df["Your Academic Stage"]
    out["presion_pares"] = df["Peer pressure"]
    out["presion_hogar"] = df["Academic pressure from your home"]
    out["entorno_estudio"] = df["Study Environment"]
    out["estrategia_afrontamiento"] = df["What coping strategy you use as a student?"].astype(str).str[:120]
    out["habitos_nocivos"] = df["Do you have any bad habits like smoking, drinking on a daily basis?"].astype(str).str[:30]
    out["competencia_academica"] = df[[c for c in df.columns if "competition" in c][0]]
    out["indice_estres"] = df[T]
    cargar("b1_academic_stress", out)

**B1.2 Anxiety GAD-7** (13.464 filas). Nivel de estrés por umbrales clínicos GAD-7 (0-4 / 5-9 / ≥10). Lectura con `encoding='latin1'`.

In [ ]:
if OMITIR_CARGA:
    print('  (carga omitida: datos ya presentes)')
else:
    df = pd.read_csv(os.path.join(BASE, "Bloque 1", "1 Student anxiety dataset",
                                  "anxiety.csv"), encoding="latin1")
    def gad_a_nivel(v):
        try: x = float(v)
        except (TypeError, ValueError): return "No aplica"
        if x <= 4: return "Bajo"
        if x <= 9: return "Moderado"
        return "Alto"
    edu_anx = df["Degree"].map({
        "Bachelor\xa0(or equivalent)": "Grado universitario",
        "Master\xa0(or equivalent)": "Posgrado",
        "Ph.D., Psy. D., MD\xa0(or equivalent)": "Posgrado",
        "High school diploma (or equivalent)": "Secundaria o menos",
        "None": "Secundaria o menos"})
    out = conformes(df, genero=df["Gender"], edad=df["Age"], edu=edu_anx,
                    estres=df["GAD_T"].map(gad_a_nivel))
    out["id_dataset"] = 2
    out["num_secuencial_csv"] = df["S. No."]
    for i in range(1, 8):
        out[f"gad{i}"] = df[f"GAD{i}"]
    out["gade"] = df["GADE"].astype(str).str[:60]
    out["swl_total"] = df["SWL_T"]
    out["spin_total"] = df["SPIN_T"]
    out["gad_total"] = df["GAD_T"]
    out["horas_juego"] = pd.to_numeric(df["Hours"], errors="coerce").clip(upper=9999)
    out["edad"] = df["Age"].map(edad_o_none)
    out["ocupacion"] = df["Work"].astype(str).str[:60]
    out["pais_residencia"] = df["Residence_ISO3"].astype(str).str[:3]
    cargar("b1_anxiety", out)

**B1.3 Mental Health Survey** (87 filas). Ojo: la columna `academic_workload` trae un espacio final en el CSV — se limpia con `strip()` al cargar.

In [ ]:
if OMITIR_CARGA:
    print('  (carga omitida: datos ya presentes)')
else:
    df = pd.read_csv(os.path.join(BASE, "Bloque 1",
                                  "1 Student Mental Health Survey/MentalHealthSurvey.csv"))
    df.columns = [c.strip() for c in df.columns]   # 'academic_workload ' trae espacio final
    edu_ms = df["degree_level"].map({"Undergraduate": "Grado universitario",
                                      "Postgraduate": "Posgrado"})
    out = conformes(df, genero=df["gender"], edad=df["age"], edu=edu_ms,
                    estres=df["depression"].map(likert_a_nivel))
    out["id_dataset"] = 3
    out["edad"] = df["age"].map(edad_o_none)
    out["universidad"] = df["university"].astype(str).str[:60]
    out["carrera"] = df["degree_major"].astype(str).str[:60]
    out["anio_academico"] = df["academic_year"].astype(str).str[:20]
    out["cgpa_rango"] = df["cgpa"].astype(str).str[:20]
    out["satisfaccion_estudios"] = df["study_satisfaction"]
    out["carga_academica"] = df["academic_workload"]
    out["presion_academica"] = df["academic_pressure"]
    out["aislamiento"] = df["isolation"]
    out["inseguridad_futuro"] = df["future_insecurity"]
    out["depresion"] = df["depression"]
    out["ansiedad"] = df["anxiety"]
    cargar("b1_mental_health_survey", out)

**B1.4 Student Habits** (1.000 filas). Dataset sintético: se carga para la capa descriptiva. `mental_health_rating` está invertida (más alto = mejor), por eso el mapeo también se invierte.

In [ ]:
if OMITIR_CARGA:
    print('  (carga omitida: datos ya presentes)')
else:
    df = pd.read_csv(os.path.join(BASE, "Bloque 1",
        "1 Student-Habits-vs-Academic-Performance", "student_habits_performance.csv"))
    def salud_inv(v):
        try: x = float(v)
        except (TypeError, ValueError): return "No aplica"
        if x >= 8: return "Bajo"
        if x >= 5: return "Moderado"
        return "Alto"
    out = conformes(df, genero=df["gender"], edad=df["age"],
                    estres=df["mental_health_rating"].map(salud_inv))
    out["id_dataset"] = 4
    out["id_estudiante"] = df["student_id"]
    out["edad"] = df["age"].map(edad_o_none)
    out["horas_estudio_dia"] = df["study_hours_per_day"]
    out["horas_redes"] = df["social_media_hours"]
    out["horas_streaming"] = df["netflix_hours"]
    out["trabajo_parcial"] = df["part_time_job"]
    out["asistencia_pct"] = df["attendance_percentage"]
    out["horas_sueno"] = df["sleep_hours"]
    out["calidad_dieta"] = df["diet_quality"].astype(str).str[:20]
    out["frecuencia_ejercicio"] = df["exercise_frequency"]
    out["educacion_padres"] = df["parental_education_level"].astype("string").str[:30]
    out["calidad_internet"] = df["internet_quality"].astype(str).str[:20]
    out["salud_mental"] = df["mental_health_rating"]
    out["nota_examen"] = df["exam_score"].clip(0, 100)
    cargar("b1_student_habits", out)

**B2.1 Social Media/Sleep** (11.000 filas). Sintético. Sin columna de género en el origen → *No reportado*.

In [ ]:
if OMITIR_CARGA:
    print('  (carga omitida: datos ya presentes)')
else:
    df = pd.read_csv(os.path.join(BASE, "Bloque 2",
        "2 Social Media Usage vs Sleep, Stress & Productivity",
        "social_media_sleep_stress_productivity_11000.csv"))
    out = conformes(df, edad=df["age"],
                    estres=df["stress_level"].map({"Low": "Bajo", "Medium": "Moderado",
                                                    "High": "Alto"}))
    out["id_dataset"] = 5
    out["id_usuario_red"] = df["user_id"]
    out["edad"] = df["age"].map(edad_o_none)
    out["plataforma"] = df["platform"].astype(str).str[:30]
    out["horas_redes"] = df["social_media_hours"]
    out["horas_pantalla"] = df["daily_screen_time_hours"].clip(0, 24)
    out["horas_sueno"] = df["sleep_hours"].clip(0, 24)
    out["horas_estudio_trabajo"] = df["study_work_hours"]
    out["minutos_ejercicio"] = df["exercise_minutes"]
    out["productividad"] = df["productivity_score"]
    out["nivel_estres_nativo"] = df["stress_level"]
    cargar("b2_social_media_sleep", out)

**B2.2 SMMH** (481 filas). Las columnas se acceden por posición porque sus nombres son preguntas completas de la encuesta.

In [ ]:
if OMITIR_CARGA:
    print('  (carga omitida: datos ya presentes)')
else:
    df = pd.read_csv(os.path.join(BASE, "Bloque 2",
                                  "2 Social Media and Mental Health/smmh.csv"))
    c = list(df.columns)
    lik = lambda i: pd.to_numeric(df[c[i]], errors="coerce")
    out = conformes(df, genero=df[c[2]], edad=df[c[1]],
                    estres=lik(18).map(likert_a_nivel))
    out["id_dataset"] = 6
    out["ts_respuesta"] = df[c[0]].astype(str).str[:30]
    out["edad"] = df[c[1]].map(edad_o_none)
    out["estado_relacion"] = df[c[3]].astype(str).str[:30]
    out["ocupacion"] = df[c[4]].astype(str).str[:30]
    out["usa_redes"] = df[c[6]].map(lambda v: "Yes" if str(v).lower().startswith("y") else "No")
    out["plataformas"] = df[c[7]].astype(str).str[:255]
    out["tiempo_diario"] = df[c[8]].astype(str).str[:40]
    campos = ["uso_sin_proposito", "distraccion_redes", "inquietud_sin_redes",
              "distraccion_facil", "preocupacion", "dificultad_concentracion",
              "comparacion_social", "sentimiento_comparacion", "busqueda_validacion",
              "se_siente_deprimido", "fluctuacion_interes", "problemas_sueno"]
    for campo, idx in zip(campos, range(9, 21)):
        out[campo] = lik(idx)
    cargar("b2_smmh", out)

**B2.3 Lifestyle** (3.000 filas). Sintético. Los NaN de condición de salud mental se recodifican como 'Ninguna'.

In [ ]:
if OMITIR_CARGA:
    print('  (carga omitida: datos ya presentes)')
else:
    df = pd.read_csv(os.path.join(BASE, "Bloque 2",
        "2 Mental_Health_Lifestyle GITHUB/Mental_Health_Lifestyle_Dataset.csv"))
    out = conformes(df, genero=df["Gender"], edad=df["Age"],
                    estres=df["Stress Level"].map({"Low": "Bajo", "Moderate": "Moderado",
                                                    "High": "Alto"}))
    out["id_dataset"] = 7
    out["pais"] = df["Country"].astype(str).str[:30]
    out["edad"] = df["Age"].map(edad_o_none)
    out["nivel_ejercicio"] = df["Exercise Level"].astype(str).str[:20]
    out["tipo_dieta"] = df["Diet Type"].astype(str).str[:30]
    out["horas_sueno"] = df["Sleep Hours"].clip(0, 24)
    out["nivel_estres_nativo"] = df["Stress Level"]
    out["condicion_salud_mental"] = df["Mental Health Condition"].fillna("Ninguna").astype(str).str[:30]
    out["horas_trabajo_semana"] = df["Work Hours per Week"].clip(0, 99)
    out["horas_pantalla"] = df["Screen Time per Day (Hours)"]
    out["interaccion_social"] = df["Social Interaction Score"]
    out["felicidad"] = df["Happiness Score"]
    cargar("b2_lifestyle", out)

**B3.1 Dreaddit train + test** (2.834 + 715 filas). Aplica la regla de calidad R7: excluye los 4 fragmentos corruptos con texto `#NAME?`. Ambas particiones van a la misma tabla diferenciadas por el campo `particion`.

In [ ]:
if OMITIR_CARGA:
    print('  (carga omitida: datos ya presentes)')
else:
    for part, ds_id in [("train", 8), ("test", 9)]:
        df = pd.read_csv(os.path.join(BASE, "Bloque 3",
            "3 Stress Analysis in Social Media Dataset", f"dreaddit-{part}.csv"))
        df = df[~df["text"].str.strip().isin(["#NAME?", "#REF!", "#VALUE!"])]  # R7
        out = conformes(df, estres=df["label"].map({0: "Bajo", 1: "Alto"}))
        out["id_dataset"] = ds_id
        out["id_fragmento_post"] = df["id"]
        out["id_subreddit"] = df["subreddit"].map(ID_SUBRED)
        out["particion"] = part
        out["post_id_origen"] = df["post_id"].astype(str).str[:20]
        out["rango_frases"] = df["sentence_range"].astype(str).str[:20]
        out["texto"] = df["text"]
        out["etiqueta_estres"] = df["label"]
        out["confianza_etiqueta"] = df["confidence"]
        out["ts_publicacion"] = pd.to_numeric(df["social_timestamp"], errors="coerce")
        out["karma"] = pd.to_numeric(df["social_karma"], errors="coerce")
        out["num_comentarios"] = pd.to_numeric(df["social_num_comments"], errors="coerce")
        out["sentimiento"] = df["sentiment"]
        out["liwc_ansiedad"] = df["lex_liwc_anx"]
        out["liwc_emo_negativa"] = df["lex_liwc_negemo"]
        out["liwc_emo_positiva"] = df["lex_liwc_posemo"]
        out["liwc_primera_persona"] = df["lex_liwc_i"]
        out["legibilidad_fk"] = df["syntax_fk_grade"]
        cargar("b3_dreaddit_post", out)

**B3.2 Depression Student** (502 filas).

In [ ]:
if OMITIR_CARGA:
    print('  (carga omitida: datos ya presentes)')
else:
    df = pd.read_csv(os.path.join(BASE, "Bloque 3",
        "3 Student_Mental_Health_Analysis GITHUB/Depression+Student+Dataset.csv"))
    out = conformes(df, genero=df["Gender"], edad=df["Age"],
                    estres=df["Depression"].map({"No": "Bajo", "Yes": "Alto"}))
    out["id_dataset"] = 10
    out["edad"] = df["Age"].map(edad_o_none)
    out["presion_academica"] = df["Academic Pressure"]
    out["satisfaccion_estudios"] = df["Study Satisfaction"]
    out["duracion_sueno"] = df["Sleep Duration"].astype(str).str[:30]
    out["habitos_alimentarios"] = df["Dietary Habits"].astype(str).str[:30]
    out["pensamientos_suicidas"] = df["Have you ever had suicidal thoughts ?"]
    out["horas_estudio"] = df["Study Hours"]
    out["estres_financiero"] = df["Financial Stress"]
    out["antecedentes_familiares"] = df["Family History of Mental Illness"]
    out["depresion"] = df["Depression"]
    cargar("b3_depression_student", out)

**B4.1 DASS-42** (39.767 filas tras filtrar edades inválidas). Calcula las tres subescalas con la clave oficial del codebook y aplica los baremos DASS para el nivel conforme. La carga opcional de `b4_dass_item` (1,67 M de filas) está desactivada por defecto.

In [ ]:
if OMITIR_CARGA:
    print('  (carga omitida: datos ya presentes)')
else:
    df = pd.read_csv(os.path.join(BASE, "Bloque 4",
        "4 Depression Anxiety Stress Scales Responses/data.csv"),
        sep="\t", low_memory=False)
    df = df[(df["age"] >= 13) & (df["age"] <= 90)].reset_index(drop=True)
    dep_i = [3,5,10,13,16,17,21,24,26,31,34,37,38,42]
    anx_i = [2,4,7,9,15,19,20,23,25,28,30,36,40,41]
    str_i = [1,6,8,11,12,14,18,22,27,29,32,33,35,39]
    sub = lambda items: df[[f"Q{i}A" for i in items]].sum(axis=1)
    df["dep_s"], df["anx_s"], df["str_s"] = sub(dep_i), sub(anx_i), sub(str_i)
    def dass_nivel(v):
        if v <= 28: return "Bajo"
        if v <= 39: return "Moderado"
        return "Alto"
    edu_dass = df["education"].map({1: "Secundaria o menos", 2: "Secundaria o menos",
                                     3: "Grado universitario", 4: "Posgrado"})
    out = conformes(df, genero=df["gender"], edad=df["age"], edu=edu_dass,
                    estres=df["str_s"].map(dass_nivel))
    out["id_dataset"] = 11
    out["pais"] = df["country"].astype(str).str[:2]
    out["edad"] = df["age"].astype(int)
    out["ingles_nativo"] = df["engnat"]
    out["religion"] = df["religion"]
    out["orientacion"] = df["orientation"]
    out["estado_civil"] = df["married"]
    out["carrera"] = df["major"].astype("string").str[:120]
    out["seg_intro"] = df["introelapse"].clip(0, 4_000_000_000)
    out["seg_test"] = df["testelapse"].clip(0, 4_000_000_000)
    out["seg_encuesta"] = df["surveyelapse"].clip(0, 4_000_000_000)
    out["validez_vcl_ok"] = (df[["VCL6", "VCL9", "VCL12"]].sum(axis=1) == 0)
    out["score_depresion"] = df["dep_s"]
    out["score_ansiedad"] = df["anx_s"]
    out["score_estres"] = df["str_s"]
    cargar("b4_dass_respondente", out)

    # --------- b4_dass_item (opcional, ~1,67M filas) ------------------------------
    if CARGAR_DASS_ITEM:
        print("  Cargando b4_dass_item (esto tarda varios minutos)...")
        with engine.begin() as con:
            ids = pd.read_sql("SELECT id_respondente_dass FROM b4_dass_respondente "
                              "ORDER BY id_respondente_dass", con)
        df = df.reset_index(drop=True)
        df["id_resp"] = ids["id_respondente_dass"].values
        sub_map = ({i: "depresion" for i in dep_i} | {i: "ansiedad" for i in anx_i}
                   | {i: "estres" for i in str_i})
        bloques = []
        for i in range(1, 43):
            b = pd.DataFrame({
                "id_respondente_dass": df["id_resp"],
                "numero_item": i,
                "subescala": sub_map[i],
                "respuesta": df[f"Q{i}A"],
                "posicion_en_test": df[f"Q{i}I"],
                "ms_respuesta": pd.to_numeric(df[f"Q{i}E"], errors="coerce")
                                  .clip(0, 4_000_000_000),
            })
            bloques.append(b)
        items = pd.concat(bloques, ignore_index=True)
        items = items[items["respuesta"].between(1, 4)]
        cargar("b4_dass_item", items)

**B4.2 Mental Health** (261.328 filas). Es el dataset más grande: esta celda tarda unos minutos. Sin edad en el origen → *No reportado*.

In [ ]:
if OMITIR_CARGA:
    print('  (carga omitida: datos ya presentes)')
else:
    df = pd.read_csv(os.path.join(BASE, "Bloque 4",
                                  "4 Mental Health Dataset/Mental Health dataset1.csv"))
    out = conformes(df, genero=df["Gender"],
                    estres=df["IncreasingStress"].map({"No": "Bajo", "Maybe": "Moderado",
                                                        "Yes": "Alto"}))
    out["id_dataset"] = 12
    out["pais"] = df["Country"].astype(str).str[:40]
    out["ocupacion"] = df["Occupation"].astype(str).str[:30]
    out["autonomo"] = df["SelfEmployed"].astype("string")
    out["antecedentes_familiares"] = df["FamilyHistory"]
    out["tratamiento"] = df["Treatment"]
    out["dias_en_casa"] = df["DaysIndoors"].astype(str).str[:30]
    out["cambio_habitos"] = df["HabitsChange"].astype(str).str[:10]
    out["historial_salud_mental"] = df["MentalHealthHistory"].astype(str).str[:10]
    out["estres_creciente"] = df["IncreasingStress"].astype(str).str[:10]
    out["cambios_humor"] = df["MoodSwings"].astype(str).str[:10]
    out["debilidad_social"] = df["SocialWeakness"].astype(str).str[:10]
    out["dificultad_afrontamiento"] = df["CopingStruggles"].astype(str).str[:10]
    out["interes_trabajo"] = df["WorkInterest"].astype(str).str[:10]
    out["entrevista_salud_mental"] = df["MentalHealthInterview"].astype(str).str[:10]
    out["conoce_opciones"] = df["CareOptions"].astype(str).str[:15]
    cargar("b4_mental_health", out)

### Verificación de la carga
Filtro global del tutor sobre la estrella: **mujeres de 18-25 años con estrés Alto**, resuelto en una sola consulta que atraviesa las fuentes de los 4 bloques. Esta celda se ejecuta siempre. Resultado de referencia: 7.446 casos (6.799 · 224 · 183 · 110 · 65 · 51 · 14).

In [ ]:
# ── Verificacion final: filtro global del tutor sobre la estrella ─────────────
print("\n" + "=" * 80)
print("VERIFICACION: Mujeres 18-25 con estres Alto (via vw_hechos_estres)")
print("=" * 80)
q = """
SELECT ds.bloque, ds.dataset, COUNT(*) AS casos
FROM vw_hechos_estres h
JOIN vw_dim_dataset      ds ON ds.id_dataset     = h.id_dataset
JOIN vw_dim_genero       g  ON g.id_genero       = h.id_genero
JOIN vw_dim_rango_edad   r  ON r.id_rango_edad   = h.id_rango_edad
JOIN vw_dim_nivel_estres e  ON e.id_nivel_estres = h.id_nivel_estres
WHERE g.codigo='F' AND r.rango='18-25' AND e.nivel='Alto'
GROUP BY ds.bloque, ds.dataset ORDER BY casos DESC
"""
print(pd.read_sql(text(q), engine).to_string(index=False))
print("\nCARGA COMPLETADA.")

---
## B.2 — ETL: transformaciones y KPIs sobre el datawarehouse en estrella

> **Cómo funciona:** si falta algún requisito, la celda 1 se detiene y dice qué script ejecutar. Se puede relanzar las veces que haga falta: no duplica datos.

### Qué hace esta parte
Es un proceso **ETL de MySQL a MySQL**:
- **Extrae** de la estrella de consumo (`vw_hechos_estres`, `vw_dim_dataset`) y de las tablas ya ingestadas en la parte B.1.
- **Transforma** con Python (pandas): integración, agregación, resumen estadístico, pivote con umbrales clínicos y tabla de contingencia.
- **Carga** el resultado en las tablas del datawarehouse (`dwh_*`), que son hechos de la estrella: guardan solo claves hacia las dimensiones conformes (género, rango de edad, nivel educativo, nivel de estrés, dataset, bloque, subreddit) y sus medidas.
- **Calcula 6 KPIs** y los guarda con histórico en `dwh_kpi_resultados`.

| Paso | Transformación | Tipo | Tabla destino |
|---|---|---|---|
| T1 | Registros de las 9 fuentes con señal real | Integración | `dwh_hechos_estres_unificado` |
| T2 | Prevalencia de estrés por bloque × género × edad | Agregación | `dwh_agg_prevalencia` |
| T3 | Señales lingüísticas de Dreaddit por subreddit y nivel | Resumen estadístico | `dwh_nlp_senales` |
| T4 | DASS por género × edad × nivel educativo | Pivote + umbrales clínicos | `dwh_dass_severidad` |
| T5 | Tratamiento según estrés creciente y antecedentes | Contingencia | `dwh_brecha_tratamiento` |

**Requisitos previos:** `modelo_datos_mysql_v2.sql`, `esquema_estrella_powerbi.sql` y la parte B.1 (ingesta). **El datawarehouse (`dwh_estrella.sql`) no hay que ejecutarlo a mano:** si falta alguna de sus tablas, la celda 1 lo crea sola.

### Celda 1 — Configuración, conexión y comprobación de requisitos

In [ ]:
import os, json
import numpy as np
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine, text

DB_USER, DB_PASS = "root", ""
DB_HOST, DB_NAME = "localhost", "tfm_estres_estudiantil_2"
engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}?charset=utf8mb4")
EJECUCION = datetime.now()

def existe(objeto):
    """True si la tabla o vista existe (sirve para ambas)."""
    try:
        with engine.connect() as con:
            con.execute(text(f"SELECT 1 FROM {objeto} LIMIT 1"))
        return True
    except Exception:
        return False

# Todo lo que este notebook lee o escribe debe existir; si no, se dice exactamente que ejecutar.
REQUISITOS = {
    "modelo_datos_mysql_v2.sql + notebook 07 (datos ingestados)": ["b3_dreaddit_post", "b3_depression_student", "b4_dass_respondente", "b4_mental_health"],
    "esquema_estrella_powerbi.sql (vistas de la estrella)": ["vw_hechos_estres", "vw_dim_dataset"],
    "modelo_datos_mysql_v2.sql (dimensiones conformes)": ["dim_dataset", "dim_bloque", "dim_genero", "dim_rango_edad", "dim_nivel_educativo", "dim_nivel_estres", "dim_subreddit"],
}
faltan = {k: [t for t in v if not existe(t)] for k, v in REQUISITOS.items()}
faltan = {k: v for k, v in faltan.items() if v}
if faltan:
    raise RuntimeError("Antes de continuar:\n" + "\n".join(f"  - falta {', '.join(v)}  ->  ejecuta: {k}" for k, v in faltan.items()))
with engine.connect() as con:
    n_estrella = con.execute(text("SELECT COUNT(*) FROM vw_hechos_estres")).scalar()
if n_estrella == 0:
    raise RuntimeError("La estrella no tiene datos: ejecuta antes el notebook 07 (carga de datos).")

# ── Datawarehouse: si falta alguna tabla (p. ej. no se ejecuto dwh_estrella.sql) se crea aqui, con el mismo script ──
DWH_TABLAS = ["dwh_hechos_estres_unificado", "dwh_agg_prevalencia", "dwh_nlp_senales", "dwh_dass_severidad",
              "dwh_brecha_tratamiento", "dwh_kpi_catalogo", "dwh_kpi_resultados", "dwh_modelo_resultados"]
DDL_DWH = """
SET FOREIGN_KEY_CHECKS = 0;
DROP VIEW  IF EXISTS vw_modelo_resultados;
DROP TABLE IF EXISTS dwh_modelo_resultados, dwh_kpi_resultados, dwh_kpi_catalogo,
                     dwh_brecha_tratamiento, dwh_dass_severidad, dwh_nlp_senales,
                     dwh_agg_prevalencia, dwh_hechos_estres_unificado;
SET FOREIGN_KEY_CHECKS = 1;

-- ============================================================================
-- T1  HECHO CENTRAL  (grano: 1 registro de las 9 fuentes con senal real)
-- ============================================================================
CREATE TABLE dwh_hechos_estres_unificado (
  id_hecho            INT UNSIGNED NOT NULL AUTO_INCREMENT,
  id_dataset          TINYINT UNSIGNED NOT NULL,
  id_genero           TINYINT UNSIGNED NOT NULL,
  id_rango_edad       TINYINT UNSIGNED NOT NULL,
  id_nivel_educativo  TINYINT UNSIGNED NOT NULL,
  id_nivel_estres     TINYINT UNSIGNED NOT NULL,
  demografia_completa BOOLEAN NOT NULL COMMENT '1 si genero y edad son reportados',
  PRIMARY KEY (id_hecho),
  CONSTRAINT fk_t1_dataset FOREIGN KEY (id_dataset)         REFERENCES dim_dataset(id_dataset),
  CONSTRAINT fk_t1_genero  FOREIGN KEY (id_genero)          REFERENCES dim_genero(id_genero),
  CONSTRAINT fk_t1_edad    FOREIGN KEY (id_rango_edad)      REFERENCES dim_rango_edad(id_rango_edad),
  CONSTRAINT fk_t1_edu     FOREIGN KEY (id_nivel_educativo) REFERENCES dim_nivel_educativo(id_nivel_educativo),
  CONSTRAINT fk_t1_estres  FOREIGN KEY (id_nivel_estres)    REFERENCES dim_nivel_estres(id_nivel_estres)
) ENGINE=InnoDB;

-- ============================================================================
-- T2  HECHO AGREGADO  (grano: bloque x genero x rango de edad)
-- ============================================================================
CREATE TABLE dwh_agg_prevalencia (
  id_agg            INT UNSIGNED NOT NULL AUTO_INCREMENT,
  id_bloque         TINYINT UNSIGNED NOT NULL,
  id_genero         TINYINT UNSIGNED NOT NULL,
  id_rango_edad     TINYINT UNSIGNED NOT NULL,
  n_total           INT UNSIGNED NOT NULL,
  n_estres_alto     INT UNSIGNED NOT NULL,
  n_estres_moderado INT UNSIGNED NOT NULL,
  n_estres_bajo     INT UNSIGNED NOT NULL,
  pct_estres_alto   DECIMAL(5,2) NOT NULL,
  PRIMARY KEY (id_agg),
  CONSTRAINT fk_t2_bloque FOREIGN KEY (id_bloque)     REFERENCES dim_bloque(id_bloque),
  CONSTRAINT fk_t2_genero FOREIGN KEY (id_genero)     REFERENCES dim_genero(id_genero),
  CONSTRAINT fk_t2_edad   FOREIGN KEY (id_rango_edad) REFERENCES dim_rango_edad(id_rango_edad)
) ENGINE=InnoDB;

-- ============================================================================
-- T3  HECHO AGREGADO NLP  (grano: subreddit x nivel de estres)
-- ============================================================================
CREATE TABLE dwh_nlp_senales (
  id_senal            INT UNSIGNED NOT NULL AUTO_INCREMENT,
  id_subreddit        SMALLINT UNSIGNED NOT NULL,
  id_nivel_estres     TINYINT UNSIGNED NOT NULL,
  n_posts             INT UNSIGNED NOT NULL,
  sentimiento_medio   DECIMAL(7,4) NOT NULL,
  liwc_ansiedad_medio DECIMAL(7,4) NOT NULL,
  liwc_emo_neg_medio  DECIMAL(7,4) NOT NULL,
  liwc_emo_pos_medio  DECIMAL(7,4) NOT NULL,
  longitud_media_palabras DECIMAL(7,2) NOT NULL,
  PRIMARY KEY (id_senal),
  CONSTRAINT fk_t3_subreddit FOREIGN KEY (id_subreddit)    REFERENCES dim_subreddit(id_subreddit),
  CONSTRAINT fk_t3_estres    FOREIGN KEY (id_nivel_estres) REFERENCES dim_nivel_estres(id_nivel_estres)
) ENGINE=InnoDB;

-- ============================================================================
-- T4  HECHO AGREGADO DASS  (grano: genero x rango de edad x nivel educativo)
-- ============================================================================
CREATE TABLE dwh_dass_severidad (
  id_dass_sev        INT UNSIGNED NOT NULL AUTO_INCREMENT,
  id_genero          TINYINT UNSIGNED NOT NULL,
  id_rango_edad      TINYINT UNSIGNED NOT NULL,
  id_nivel_educativo TINYINT UNSIGNED NOT NULL,
  n_respondentes     INT UNSIGNED NOT NULL,
  depresion_media    DECIMAL(6,2) NOT NULL,
  ansiedad_media     DECIMAL(6,2) NOT NULL,
  estres_medio       DECIMAL(6,2) NOT NULL,
  pct_estres_severo  DECIMAL(5,2) NOT NULL COMMENT '% con estres >= 40 (baremo DASS)',
  PRIMARY KEY (id_dass_sev),
  CONSTRAINT fk_t4_genero FOREIGN KEY (id_genero)          REFERENCES dim_genero(id_genero),
  CONSTRAINT fk_t4_edad   FOREIGN KEY (id_rango_edad)      REFERENCES dim_rango_edad(id_rango_edad),
  CONSTRAINT fk_t4_edu    FOREIGN KEY (id_nivel_educativo) REFERENCES dim_nivel_educativo(id_nivel_educativo)
) ENGINE=InnoDB;

-- ============================================================================
-- T5  TABLA DE CONTINGENCIA  (grano: estres creciente x antecedentes familiares)
-- ============================================================================
CREATE TABLE dwh_brecha_tratamiento (
  id_brecha               INT UNSIGNED NOT NULL AUTO_INCREMENT,
  id_nivel_estres         TINYINT UNSIGNED NOT NULL COMMENT 'Estres creciente: No=Bajo, Maybe=Moderado, Yes=Alto',
  antecedentes_familiares ENUM('Yes','No') NOT NULL COMMENT 'Dimension degenerada',
  n_total                 INT UNSIGNED NOT NULL,
  n_con_tratamiento       INT UNSIGNED NOT NULL,
  pct_tratamiento         DECIMAL(5,2) NOT NULL,
  PRIMARY KEY (id_brecha),
  CONSTRAINT fk_t5_estres FOREIGN KEY (id_nivel_estres) REFERENCES dim_nivel_estres(id_nivel_estres)
) ENGINE=InnoDB;

-- ============================================================================
-- KPIs: catalogo (definicion) + historico de resultados (1 fila por KPI y ejecucion)
-- ============================================================================
CREATE TABLE dwh_kpi_catalogo (
  id_kpi         TINYINT UNSIGNED NOT NULL,
  codigo         VARCHAR(30)  NOT NULL UNIQUE,
  nombre         VARCHAR(120) NOT NULL,
  descripcion    VARCHAR(400) NOT NULL,
  unidad         VARCHAR(20)  NOT NULL,
  fuente_calculo VARCHAR(60)  NOT NULL,
  PRIMARY KEY (id_kpi)
) ENGINE=InnoDB;

INSERT INTO dwh_kpi_catalogo VALUES
 (1,'KPI_ESTRES_ALTO','Tasa de estres Alto global',
    'Porcentaje de registros en nivel de estres Alto sobre el total de las 9 fuentes con senal real.','%','dwh_hechos_estres_unificado'),
 (2,'KPI_BRECHA_GENERO','Brecha de genero en ansiedad (DASS)',
    'Diferencia entre la ansiedad media de mujeres y hombres en la escala DASS-42 (18-25 anos).','puntos','b4_dass_respondente'),
 (3,'KPI_RIESGO_CRITICO','Indice de riesgo critico',
    'Porcentaje de estudiantes con pensamientos suicidas que ademas presentan depresion.','%','b3_depression_student'),
 (4,'KPI_TASA_TRATAMIENTO','Tasa de busqueda de tratamiento',
    'Porcentaje de estudiantes que han buscado tratamiento psicologico.','%','dwh_brecha_tratamiento'),
 (5,'KPI_DISCRIMINANTE_LIWC','Poder discriminante LIWC',
    'Ratio entre la densidad media de palabras de ansiedad (LIWC) de la clase estres y la de no-estres en Dreaddit.','ratio','b3_dreaddit_post'),
 (6,'KPI_COBERTURA_DEMO','Cobertura demografica',
    'Porcentaje de registros con genero y edad reportados sobre el total unificado.','%','dwh_hechos_estres_unificado');

CREATE TABLE dwh_kpi_resultados (
  id_resultado    INT UNSIGNED NOT NULL AUTO_INCREMENT,
  id_kpi          TINYINT UNSIGNED NOT NULL,
  codigo          VARCHAR(30)   NOT NULL,
  valor           DECIMAL(12,4) NOT NULL,
  detalle         VARCHAR(255)  NULL,
  fecha_ejecucion DATETIME      NOT NULL,
  PRIMARY KEY (id_resultado),
  CONSTRAINT fk_kpires_catalogo FOREIGN KEY (id_kpi) REFERENCES dwh_kpi_catalogo(id_kpi),
  INDEX idx_kpires_fecha (fecha_ejecucion)
) ENGINE=InnoDB;

-- ============================================================================
-- OBJETIVO 2: resultados de modelado (grano: dataset x modelo x ejecucion)
-- Comparte con la estrella la dimension dim_dataset (via id_dataset).
-- ============================================================================
CREATE TABLE dwh_modelo_resultados (
  id_resultado_modelo INT UNSIGNED NOT NULL AUTO_INCREMENT,
  id_dataset          TINYINT UNSIGNED NOT NULL COMMENT 'Dreaddit (train+test) se registra con id 8',
  tarea_modelado      VARCHAR(30)  NOT NULL,
  modelo              VARCHAR(60)  NOT NULL,
  protocolo           VARCHAR(25)  NOT NULL,
  n_registros         INT UNSIGNED NOT NULL,
  accuracy            DECIMAL(6,4) NOT NULL,
  f1                  DECIMAL(6,4) NOT NULL,
  auc                 DECIMAL(6,4) NOT NULL,
  auc_desv            DECIMAL(6,4) NULL COMMENT 'Solo en validacion cruzada',
  brecha_acc          DECIMAL(7,4) NOT NULL COMMENT 'Accuracy train - test (sobreajuste)',
  es_mejor_modelo     BOOLEAN NOT NULL,
  target              VARCHAR(80)  NOT NULL,
  lectura_capa_armonizada VARCHAR(160) NOT NULL,
  alerta              VARCHAR(160) NULL,
  fecha_ejecucion     DATETIME NOT NULL,
  PRIMARY KEY (id_resultado_modelo),
  CONSTRAINT fk_modres_dataset FOREIGN KEY (id_dataset) REFERENCES dim_dataset(id_dataset),
  INDEX idx_modres_ejecucion (fecha_ejecucion)
) ENGINE=InnoDB;

-- Vista de consumo: resultados de la ULTIMA ejecucion con la dimension dataset incorporada
CREATE OR REPLACE VIEW vw_modelo_resultados AS
SELECT r.id_resultado_modelo, r.id_dataset, ds.bloque, ds.dataset, ds.calidad, ds.uso_modelado,
       r.modelo, r.protocolo, r.n_registros, r.accuracy, r.f1, r.auc, r.auc_desv, r.brecha_acc,
       r.es_mejor_modelo, r.target, r.lectura_capa_armonizada, r.alerta, r.fecha_ejecucion
FROM dwh_modelo_resultados r
JOIN vw_dim_dataset ds ON ds.id_dataset = r.id_dataset
WHERE r.fecha_ejecucion = (SELECT MAX(fecha_ejecucion) FROM dwh_modelo_resultados);
"""
falta_dwh = [t for t in DWH_TABLAS if not existe(t)]
if falta_dwh:
    print("Faltan tablas del datawarehouse:", ", ".join(falta_dwh))
    print("-> se crea el datawarehouse completo (mismo contenido que dwh_estrella.sql; las tablas quedan vacías)...")
    sentencias = [s.strip() for s in "\n".join(l for l in DDL_DWH.splitlines() if not l.strip().startswith("--")).split(";") if s.strip()]
    with engine.begin() as con:
        for s in sentencias:
            con.execute(text(s))
    falta_dwh = [t for t in DWH_TABLAS if not existe(t)]
    if falta_dwh:
        raise RuntimeError("No se pudo crear el datawarehouse. Faltan: " + ", ".join(falta_dwh))
    print("Datawarehouse creado: 8 tablas y la vista vw_modelo_resultados.")
else:
    print("Datawarehouse: las 8 tablas existen.")

def nombres(tabla, id_col, nom_col):
    """Diccionario id -> nombre de una dimension (solo para mostrar tablas legibles)."""
    return dict(pd.read_sql(text(f"SELECT {id_col}, {nom_col} FROM {tabla}"), engine).itertuples(index=False))
GEN, RNG = nombres("dim_genero", "id_genero", "nombre"), nombres("dim_rango_edad", "id_rango_edad", "rango")
EDU, EST = nombres("dim_nivel_educativo", "id_nivel_educativo", "nivel"), nombres("dim_nivel_estres", "id_nivel_estres", "nivel")
BLQ, SUB = nombres("dim_bloque", "id_bloque", "nombre"), nombres("dim_subreddit", "id_subreddit", "nombre")

print(f"Conectado a {DB_NAME} | estrella: {n_estrella:,} registros | ejecución: {EJECUCION:%Y-%m-%d %H:%M:%S}")

### Celda 2 — Limpieza previa (se puede relanzar sin duplicar datos)
Las cinco tablas de transformación se vacían al empezar. Las tablas de KPIs **no**: `dwh_kpi_resultados` conserva el histórico (cada ejecución añade 6 filas con su fecha).

In [ ]:
if "engine" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (configuración y conexión). Usa Kernel > Restart Kernel and Run All Cells.")
TABLAS_T = ["dwh_hechos_estres_unificado", "dwh_agg_prevalencia", "dwh_nlp_senales", "dwh_dass_severidad", "dwh_brecha_tratamiento"]
with engine.begin() as con:
    for t in TABLAS_T:
        n = con.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar()
        con.execute(text(f"DELETE FROM {t}"))
        print(f"  {t:<30} {n:>8} filas previas eliminadas")
print("Tablas de transformación listas para una carga limpia.")

---
## T1 — Registros unificados (integración)
**Extrae** de la estrella los registros de las nueve fuentes con señal real (`uso_modelado = 'incluido'`; los tres datasets sintéticos quedan fuera, como acordó el tutor). **Transforma** añadiendo `demografia_completa` (género y edad reportados). **Carga** el hecho central del datawarehouse, que guarda solo claves hacia las dimensiones conformes.

In [ ]:
if "engine" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (configuración y conexión). Usa Kernel > Restart Kernel and Run All Cells.")
q = """SELECT h.id_dataset, h.id_genero, h.id_rango_edad, h.id_nivel_educativo, h.id_nivel_estres
       FROM vw_hechos_estres h
       JOIN vw_dim_dataset ds ON ds.id_dataset = h.id_dataset
       WHERE ds.uso_modelado = 'incluido'"""
T1 = pd.read_sql(text(q), engine)
T1["demografia_completa"] = ((T1["id_genero"] != 0) & (T1["id_rango_edad"] != 0)).astype(int)
T1.to_sql("dwh_hechos_estres_unificado", engine, if_exists="append", index=False, chunksize=5000)
print(f"T1: {len(T1):,} registros -> dwh_hechos_estres_unificado")
bloques = pd.read_sql(text("SELECT d.id_dataset, b.nombre AS bloque FROM dim_dataset d JOIN dim_bloque b ON b.id_bloque = d.id_bloque"), engine)
display(T1.merge(bloques, on="id_dataset")["bloque"].value_counts().rename("registros").to_frame())

**Opinión.** La validación psicológica aporta el 94 % de los registros (301.095 de 319.318): da robustez estadística, pero inclina el volumen hacia población general; los bloques más estudiantiles suman solo el 6 %. Por eso cualquier lectura por bloque debe acompañarse de su tamaño muestral.

---
## T2 — Prevalencia de estrés (agregación)
Agrupa T1 por **bloque × género × rango de edad** y cuenta cuántos registros hay en cada nivel de estrés. Es un hecho agregado: Power BI lo consume sin recalcular y lo filtra con las mismas dimensiones que el resto.

In [ ]:
if "engine" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (configuración y conexión). Usa Kernel > Restart Kernel and Run All Cells.")
if "T1" not in globals():
    raise RuntimeError("Falta ejecutar antes la celda que define T1. Usa Kernel > Restart Kernel and Run All Cells.")
ds_bloque = pd.read_sql(text("SELECT id_dataset, id_bloque FROM dim_dataset"), engine)
T2 = (T1.merge(ds_bloque, on="id_dataset")
        .groupby(["id_bloque", "id_genero", "id_rango_edad"])
        .agg(n_total=("id_nivel_estres", "size"),
             n_estres_alto=("id_nivel_estres", lambda s: int((s == 3).sum())),
             n_estres_moderado=("id_nivel_estres", lambda s: int((s == 2).sum())),
             n_estres_bajo=("id_nivel_estres", lambda s: int((s == 1).sum())))
        .reset_index())
T2["pct_estres_alto"] = (T2["n_estres_alto"] / T2["n_total"] * 100).round(2)
T2.to_sql("dwh_agg_prevalencia", engine, if_exists="append", index=False)
print(f"T2: {len(T2)} grupos -> dwh_agg_prevalencia")
vista = T2.assign(bloque=T2.id_bloque.map(BLQ), genero=T2.id_genero.map(GEN), rango_edad=T2.id_rango_edad.map(RNG))
display(vista.sort_values("pct_estres_alto", ascending=False)[["bloque", "genero", "rango_edad", "n_total", "pct_estres_alto"]].head(8))

**Opinión.** En los 12 grupos con más de 1.000 registros, el porcentaje de estrés Alto va del 14,7 % al 52,3 %: bloque, género y edad sí discriminan. Cuidado con los grupos pequeños: 18 de los 49 tienen menos de 30 registros y sus porcentajes extremos (100 %) no son interpretables. El dashboard debe mostrar siempre `n_total` junto al porcentaje.

---
## T3 — Señales lingüísticas de Dreaddit (resumen estadístico)
Resume los ~3.500 fragmentos de texto en una tabla compacta: por **subreddit y nivel de estrés** (Bajo = no estrés, Alto = estrés) calcula el sentimiento, las categorías LIWC de ansiedad y emociones, y la longitud media en palabras.

In [ ]:
if "engine" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (configuración y conexión). Usa Kernel > Restart Kernel and Run All Cells.")
if "T1" not in globals():
    raise RuntimeError("Falta ejecutar antes la celda que define T1. Usa Kernel > Restart Kernel and Run All Cells.")
d = pd.read_sql(text("""SELECT id_subreddit, id_nivel_estres, sentimiento, liwc_ansiedad, liwc_emo_negativa, liwc_emo_positiva, texto
                        FROM b3_dreaddit_post"""), engine)
for c in ["sentimiento", "liwc_ansiedad", "liwc_emo_negativa", "liwc_emo_positiva"]:
    d[c] = d[c].astype(float)
d["n_words"] = d["texto"].str.split().str.len()
T3 = (d.groupby(["id_subreddit", "id_nivel_estres"])
        .agg(n_posts=("texto", "size"), sentimiento_medio=("sentimiento", "mean"),
             liwc_ansiedad_medio=("liwc_ansiedad", "mean"), liwc_emo_neg_medio=("liwc_emo_negativa", "mean"),
             liwc_emo_pos_medio=("liwc_emo_positiva", "mean"), longitud_media_palabras=("n_words", "mean"))
        .round(4).reset_index())
T3.to_sql("dwh_nlp_senales", engine, if_exists="append", index=False)
print(f"T3: {len(T3)} combinaciones subreddit x nivel -> dwh_nlp_senales")
display(T3.assign(subreddit=T3.id_subreddit.map(SUB), nivel=T3.id_nivel_estres.map(EST))
          [["subreddit", "nivel", "n_posts", "sentimiento_medio", "liwc_ansiedad_medio"]].head(8))

**Opinión.** En los 10 subreddits, los textos con estrés tienen más palabras de ansiedad (LIWC) y un sentimiento más bajo que los textos sin estrés de la misma comunidad: la señal no es un artefacto de un subreddit concreto. Esto avala usar las variables LIWC en el modelado de lenguaje del Objetivo 2.

---
## T4 — DASS por demografía con severidad clínica (pivote + umbrales)
Toma los respondentes de 18 a 25 años (proxy de población estudiantil) y calcula, por **género × nivel educativo**, las medias de las tres subescalas y el porcentaje en estrés **severo** (puntuación ≥ 40, baremo DASS).

In [ ]:
if "engine" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (configuración y conexión). Usa Kernel > Restart Kernel and Run All Cells.")
if "T1" not in globals():
    raise RuntimeError("Falta ejecutar antes la celda que define T1. Usa Kernel > Restart Kernel and Run All Cells.")
d = pd.read_sql(text("""SELECT id_genero, id_rango_edad, id_nivel_educativo, score_depresion, score_ansiedad, score_estres
                        FROM b4_dass_respondente WHERE id_rango_edad = 2"""), engine)
T4 = (d.groupby(["id_genero", "id_rango_edad", "id_nivel_educativo"])
        .agg(n_respondentes=("score_estres", "size"), depresion_media=("score_depresion", "mean"),
             ansiedad_media=("score_ansiedad", "mean"), estres_medio=("score_estres", "mean"),
             pct_estres_severo=("score_estres", lambda s: (s >= 40).mean() * 100))
        .round(2).reset_index())
T4.to_sql("dwh_dass_severidad", engine, if_exists="append", index=False)
print(f"T4: {len(T4)} grupos demográficos -> dwh_dass_severidad")
display(T4.assign(genero=T4.id_genero.map(GEN), educacion=T4.id_nivel_educativo.map(EDU))
          .sort_values("estres_medio", ascending=False)
          [["genero", "educacion", "n_respondentes", "estres_medio", "pct_estres_severo"]].head(6))

**Opinión.** El grupo «Otro» con estudios de secundaria o menos concentra el malestar más alto (estrés medio 38,4 y 50 % en rango severo, n = 156), por encima de las mujeres con el mismo nivel educativo (36,5). Tanto en mujeres como en «Otro», el nivel de secundaria o menos supera al universitario. Nota: «Otro» incluye 37 respondentes de 18-25 años que no contestaron el género (código 0 del DASS), clasificados así por la ingesta.

---
## T5 — Brecha de tratamiento (tabla de contingencia)
Cruza, para los estudiantes del dataset de salud mental, el **estrés creciente** (Bajo / Moderado / Alto) con los **antecedentes familiares** y cuenta cuántos han buscado tratamiento. Es una contingencia de dos dimensiones: una conformada (nivel de estrés) y otra degenerada (antecedentes).

In [ ]:
if "engine" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (configuración y conexión). Usa Kernel > Restart Kernel and Run All Cells.")
if "T1" not in globals():
    raise RuntimeError("Falta ejecutar antes la celda que define T1. Usa Kernel > Restart Kernel and Run All Cells.")
d = pd.read_sql(text("""SELECT id_nivel_estres, antecedentes_familiares, tratamiento FROM b4_mental_health
                        WHERE ocupacion = 'Student' AND antecedentes_familiares IS NOT NULL AND tratamiento IS NOT NULL"""), engine)
T5 = (d.groupby(["id_nivel_estres", "antecedentes_familiares"])
        .agg(n_total=("tratamiento", "size"), n_con_tratamiento=("tratamiento", lambda s: int((s == "Yes").sum())))
        .reset_index())
T5["pct_tratamiento"] = (T5["n_con_tratamiento"] / T5["n_total"] * 100).round(2)
T5.to_sql("dwh_brecha_tratamiento", engine, if_exists="append", index=False)
print(f"T5: {len(T5)} celdas de la contingencia -> dwh_brecha_tratamiento")
display(T5.assign(estres_creciente=T5.id_nivel_estres.map(EST))[["estres_creciente", "antecedentes_familiares", "n_total", "pct_tratamiento"]])
por_estres = T5.groupby("id_nivel_estres")[["n_total", "n_con_tratamiento"]].sum()
por_antec = T5.groupby("antecedentes_familiares")[["n_total", "n_con_tratamiento"]].sum()
print("\n% con tratamiento según estrés creciente:  ", {EST[i]: round(r.n_con_tratamiento / r.n_total * 100, 1) for i, r in por_estres.iterrows()})
print("% con tratamiento según antecedentes:       ", {i: round(r.n_con_tratamiento / r.n_total * 100, 1) for i, r in por_antec.iterrows()})

**Opinión.** La contingencia es contundente: dentro de cada nivel de estrés, los antecedentes familiares marcan la diferencia (≈ 35 % sin antecedentes frente a ≈ 72-73 % con ellos), mientras que el nivel de estrés apenas mueve el resultado (48-51 %). Es la brecha entre malestar percibido y búsqueda de ayuda: la familiaridad con la salud mental pesa más que la intensidad del malestar.

---
## KPIs y almacenamiento con histórico
Seis indicadores calculados con Python sobre las tablas anteriores y las ingestadas. Cada ejecución inserta 6 filas en `dwh_kpi_resultados` con su fecha, de modo que el valor de un KPI nunca se sobrescribe. La definición de cada uno está en `dwh_kpi_catalogo` y en el documento `KPIs_TFM.docx`.

In [ ]:
if "engine" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (configuración y conexión). Usa Kernel > Restart Kernel and Run All Cells.")
if "T1" not in globals():
    raise RuntimeError("Falta ejecutar antes la celda que define T1. Usa Kernel > Restart Kernel and Run All Cells.")
if "T5" not in globals():
    raise RuntimeError("Falta ejecutar antes la celda que define T5. Usa Kernel > Restart Kernel and Run All Cells.")
kpis = []
# KPI 1 - Tasa de estrés Alto (sobre T1)
n_alto = int((T1["id_nivel_estres"] == 3).sum())
kpis.append((1, "KPI_ESTRES_ALTO", round(n_alto / len(T1) * 100, 4), f"{n_alto:,} de {len(T1):,} registros"))
# KPI 2 - Brecha de género en ansiedad, DASS 18-25 (mujer - hombre)
a = pd.read_sql(text("SELECT id_genero, AVG(score_ansiedad) AS a FROM b4_dass_respondente WHERE id_rango_edad = 2 AND id_genero IN (1, 2) GROUP BY id_genero"),
                engine).set_index("id_genero")["a"].astype(float)
kpis.append((2, "KPI_BRECHA_GENERO", round(a[2] - a[1], 4), f"Mujer {a[2]:.1f} - Hombre {a[1]:.1f}"))
# KPI 3 - Riesgo crítico: % de estudiantes con pensamientos suicidas que presentan depresión
k3 = float(pd.read_sql(text("SELECT 100.0 * AVG(CASE WHEN depresion = 'Yes' THEN 1.0 ELSE 0.0 END) AS v FROM b3_depression_student WHERE pensamientos_suicidas = 'Yes'"), engine).iloc[0, 0])
kpis.append((3, "KPI_RIESGO_CRITICO", round(k3, 4), "% de estudiantes con pensamientos suicidas que presentan depresión"))
# KPI 4 - Tasa de búsqueda de tratamiento (sobre T5)
kpis.append((4, "KPI_TASA_TRATAMIENTO", round(T5["n_con_tratamiento"].sum() / T5["n_total"].sum() * 100, 4), f"{int(T5['n_con_tratamiento'].sum()):,} de {int(T5['n_total'].sum()):,} estudiantes"))
# KPI 5 - Poder discriminante LIWC: ansiedad en estrés / ansiedad en no estrés
lw = pd.read_sql(text("SELECT etiqueta_estres, AVG(liwc_ansiedad) AS a FROM b3_dreaddit_post GROUP BY etiqueta_estres"), engine).set_index("etiqueta_estres")["a"].astype(float)
kpis.append((5, "KPI_DISCRIMINANTE_LIWC", round(lw[1] / lw[0], 4), f"estrés {lw[1]:.2f} / no estrés {lw[0]:.2f}"))
# KPI 6 - Cobertura demográfica (sobre T1)
kpis.append((6, "KPI_COBERTURA_DEMO", round(T1["demografia_completa"].mean() * 100, 4), "% de registros con género y edad reportados"))

res = pd.DataFrame(kpis, columns=["id_kpi", "codigo", "valor", "detalle"])
res["fecha_ejecucion"] = EJECUCION
res.to_sql("dwh_kpi_resultados", engine, if_exists="append", index=False)
print("KPIs calculados y almacenados:")
display(res[["codigo", "valor", "detalle"]])

---
## Verificación final
Tres comprobaciones sobre la base de datos:
1. **KPIs** de la última ejecución, uniendo catálogo y resultados (es la consulta que alimentará las tarjetas de Power BI).
2. **Integridad de la estrella:** ninguna clave foránea huérfana en ninguna tabla de hechos.
3. **Cuadre con la estrella:** los registros 'incluido' de `vw_hechos_estres` deben sumar lo mismo que T1.

In [ ]:
if "engine" not in globals():
    raise RuntimeError("Falta ejecutar la celda 1 (configuración y conexión). Usa Kernel > Restart Kernel and Run All Cells.")
if "T1" not in globals():
    raise RuntimeError("Falta ejecutar antes la celda que define T1. Usa Kernel > Restart Kernel and Run All Cells.")
if "res" not in globals():
    raise RuntimeError("Falta ejecutar antes la celda que define res. Usa Kernel > Restart Kernel and Run All Cells.")
q = """SELECT c.codigo, c.nombre, r.valor, c.unidad, r.detalle
       FROM dwh_kpi_resultados r JOIN dwh_kpi_catalogo c ON c.id_kpi = r.id_kpi
       WHERE r.fecha_ejecucion = (SELECT MAX(fecha_ejecucion) FROM dwh_kpi_resultados) ORDER BY c.id_kpi"""
display(pd.read_sql(text(q), engine))

CLAVES = [("dwh_hechos_estres_unificado", "id_dataset", "dim_dataset"), ("dwh_hechos_estres_unificado", "id_genero", "dim_genero"),
          ("dwh_hechos_estres_unificado", "id_rango_edad", "dim_rango_edad"), ("dwh_hechos_estres_unificado", "id_nivel_educativo", "dim_nivel_educativo"),
          ("dwh_hechos_estres_unificado", "id_nivel_estres", "dim_nivel_estres"), ("dwh_agg_prevalencia", "id_bloque", "dim_bloque"),
          ("dwh_agg_prevalencia", "id_genero", "dim_genero"), ("dwh_agg_prevalencia", "id_rango_edad", "dim_rango_edad"),
          ("dwh_nlp_senales", "id_subreddit", "dim_subreddit"), ("dwh_nlp_senales", "id_nivel_estres", "dim_nivel_estres"),
          ("dwh_dass_severidad", "id_genero", "dim_genero"), ("dwh_dass_severidad", "id_rango_edad", "dim_rango_edad"),
          ("dwh_dass_severidad", "id_nivel_educativo", "dim_nivel_educativo"), ("dwh_brecha_tratamiento", "id_nivel_estres", "dim_nivel_estres")]
filas = []
with engine.connect() as con:
    for t, col, dim in CLAVES:
        huerfanas = con.execute(text(f"SELECT COUNT(*) FROM {t} f LEFT JOIN {dim} x ON x.{col} = f.{col} WHERE x.{col} IS NULL")).scalar()
        total = con.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar()
        filas.append({"tabla de hechos": t, "clave": col, "dimensión": dim, "filas": total, "huérfanas": huerfanas})
integ = pd.DataFrame(filas)
display(integ)
print("INTEGRIDAD OK: 0 claves huérfanas." if integ["huérfanas"].sum() == 0 else "ATENCIÓN: hay claves huérfanas.")

with engine.connect() as con:
    tot, reales = con.execute(text("""SELECT COUNT(*), SUM(CASE WHEN ds.uso_modelado = 'incluido' THEN 1 ELSE 0 END)
                                      FROM vw_hechos_estres h JOIN vw_dim_dataset ds ON ds.id_dataset = h.id_dataset""")).one()
    en_dwh = con.execute(text("SELECT COUNT(*) FROM dwh_hechos_estres_unificado")).scalar()
print(f"\nEstrella: {int(tot):,} registros ({int(reales):,} de fuentes con señal real) | T1 en el DWH: {en_dwh:,}")
print("CUADRE OK: el DWH coincide con las fuentes 'incluido' de la estrella." if int(reales) == en_dwh else "ATENCIÓN: no cuadra; revisa que el notebook 07 se ejecutó una sola vez.")
print("\n✅ ETL ENTREGA 2 COMPLETADO — notebook 08 v4")

---
## Fin del notebook
Con esta ejecución quedan cubiertas las dos entregas: **Entrega 1** (EDA y calidad de los 12 datasets, con 13 figuras en `figuras1`) y **Entrega 2** (ingesta a MySQL, 5 transformaciones cargadas en tablas `dwh_*` con claves hacia las dimensiones conformes, y 6 KPIs guardados con histórico en `dwh_kpi_resultados`). La definición de cada KPI está en `KPIs_TFM.docx` y en la tabla `dwh_kpi_catalogo`.